## 1. Imports and experiment configuration

In [1]:
import os
import json
import math
import random
import zipfile
from pathlib import Path
from dataclasses import dataclass
from collections import Counter
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from time import perf_counter
import time
from tqdm.auto import tqdm
from IPython.display import clear_output, display


In [2]:
PROJECT_DIR = Path(".").resolve()
DATA_DIR = PROJECT_DIR / "data"
SPOTIFY_DIR = DATA_DIR / "spotify_mpd"
SPOTIFY_DIR.mkdir(parents=True, exist_ok=True)

SAVE_DIR = PROJECT_DIR / "processed_data"
SAVE_DIR.mkdir(parents=True, exist_ok=True)

MODEL_DIR = PROJECT_DIR / "saved_model_checkpoints"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

MPD_ZIP_PATH = None

MPD_MAX_SLICES = os.environ.get("MPD_MAX_SLICES", "None")
MPD_MAX_SLICES = None if MPD_MAX_SLICES.lower() == "none" else int(MPD_MAX_SLICES)

MIN_PLAYLIST_LENGTH = int(os.environ.get("MIN_PLAYLIST_LENGTH", "10"))
MAX_PLAYLISTS = os.environ.get("MAX_PLAYLISTS", "None")
MAX_PLAYLISTS = None if MAX_PLAYLISTS.lower() == "none" else int(MAX_PLAYLISTS)

MAX_ITEMS = os.environ.get("MAX_ITEMS", "20000")
MAX_ITEMS = None if MAX_ITEMS.lower() == "none" else int(MAX_ITEMS)
MAX_SEQ_LEN = int(os.environ.get("MAX_SEQ_LEN", "100"))

SUBSAMPLE_FRACTION = float(os.environ.get("SUBSAMPLE_FRACTION", "0.70"))
SUBSAMPLE_RANDOM_STATE = int(os.environ.get("SUBSAMPLE_RANDOM_STATE", "2026"))

D_MODEL = int(os.environ.get("D_MODEL", "256"))
N_HEADS = int(os.environ.get("N_HEADS", "8"))
N_LAYERS = int(os.environ.get("N_LAYERS", "3"))
DROPOUT = float(os.environ.get("DROPOUT", "0.1"))
GRU_HIDDEN = int(os.environ.get("GRU_HIDDEN", str(D_MODEL)))
N_INTERESTS = int(os.environ.get("N_INTERESTS", "4"))

FMLP_N_LAYERS = int(os.environ.get("FMLP_N_LAYERS", "2"))
FMLP_DROPOUT = float(os.environ.get("FMLP_DROPOUT", str(DROPOUT)))

BATCH_SIZE = int(os.environ.get("BATCH_SIZE", "1024"))
LR = float(os.environ.get("LR", "1e-3"))
WEIGHT_DECAY = float(os.environ.get("WEIGHT_DECAY", "1e-4"))
EPOCHS = int(os.environ.get("EPOCHS", "30"))
PATIENCE = int(os.environ.get("PATIENCE", "3"))

DO_CL4SREC_SSL = os.environ.get("DO_CL4SREC_SSL", "1") == "1"
DO_SIMDCL = os.environ.get("DO_SIMDCL", "1") == "1"
SSL_EPOCHS = int(os.environ.get("SSL_EPOCHS", "10"))
SSL_TAU = float(os.environ.get("SSL_TAU", "0.2"))
SIMDCL_NOISE_STD = float(os.environ.get("SIMDCL_NOISE_STD", "0.1"))

COMIREC_LABEL_AWARE = os.environ.get("COMIREC_LABEL_AWARE", "1") == "1"
COMIREC_DIVERSITY_REG = float(os.environ.get("COMIREC_DIVERSITY_REG", "0.05"))
COMIREC_LABEL_TAU = float(os.environ.get("COMIREC_LABEL_TAU", "1.0"))

TOPK = int(os.environ.get("TOPK", "20"))
EVAL_BATCH_SIZE = int(os.environ.get("EVAL_BATCH_SIZE", "1024"))
ITEM_CHUNK_SIZE = int(os.environ.get("ITEM_CHUNK_SIZE", "16384"))

NUM_EVAL_PLAYLISTS = os.environ.get("NUM_EVAL_PLAYLISTS", "None")
NUM_EVAL_PLAYLISTS = None if NUM_EVAL_PLAYLISTS.lower() == "none" else int(NUM_EVAL_PLAYLISTS)

ITEM_TITLE_MAX_TOKENS = int(os.environ.get("ITEM_TITLE_MAX_TOKENS", "8"))
PLAYLIST_TITLE_MAX_TOKENS = int(os.environ.get("PLAYLIST_TITLE_MAX_TOKENS", "8"))
TEXT_VOCAB_MAX = int(os.environ.get("TEXT_VOCAB_MAX", "20000"))

IRRT_RETRIEVAL_CUTOFFS = [50, 100, 200]
IRRT_RERANK_CANDIDATE_SIZE = int(os.environ.get("IRRT_RERANK_CANDIDATE_SIZE", "200"))
IRRT_RERANK_TOPK = int(os.environ.get("IRRT_RERANK_TOPK", str(TOPK)))
IRRT_HISTORY_TEXT_LAST_N = int(os.environ.get("IRRT_HISTORY_TEXT_LAST_N", "20"))
IRRT_TITLE_QUERY_WEIGHT = float(os.environ.get("IRRT_TITLE_QUERY_WEIGHT", "2.0"))
IRRT_HISTORY_QUERY_WEIGHT = float(os.environ.get("IRRT_HISTORY_QUERY_WEIGHT", "1.0"))
IRRT_HYBRID_DENSE_WEIGHT = float(os.environ.get("IRRT_HYBRID_DENSE_WEIGHT", "1.0"))
IRRT_HYBRID_SPARSE_WEIGHT = float(os.environ.get("IRRT_HYBRID_SPARSE_WEIGHT", "1.0"))

EXACT_DENSE_PIPELINE_NAME = "ExactDenseRetriever"
HYBRID_PIPELINE_NAME = "HybridRRF"

SEEDS = [42, 43, 44]
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

ITEM_MATRIX_CACHE: Dict[str, torch.Tensor] = {}

assert MIN_PLAYLIST_LENGTH >= 3, "Need at least 3 items per playlist for train/valid/test split."
assert 0.0 < SUBSAMPLE_FRACTION <= 1.0
assert IRRT_RERANK_CANDIDATE_SIZE >= IRRT_RERANK_TOPK, "Candidate size must be >= rerank top-k."

EXPERIMENT_CONFIG = {
    "MAX_SEQ_LEN": MAX_SEQ_LEN,
    "D_MODEL": D_MODEL,
    "N_HEADS": N_HEADS,
    "N_LAYERS": N_LAYERS,
    "DROPOUT": DROPOUT,
    "GRU_HIDDEN": GRU_HIDDEN,
    "N_INTERESTS": N_INTERESTS,
    "FMLP_N_LAYERS": FMLP_N_LAYERS,
    "FMLP_DROPOUT": FMLP_DROPOUT,
    "ITEM_TITLE_MAX_TOKENS": ITEM_TITLE_MAX_TOKENS,
    "PLAYLIST_TITLE_MAX_TOKENS": PLAYLIST_TITLE_MAX_TOKENS,
}

print("SPOTIFY_DIR:", SPOTIFY_DIR)
print("MODEL_DIR:", MODEL_DIR)
print("MPD_MAX_SLICES:", MPD_MAX_SLICES)
print("Filtering:", dict(
    MIN_PLAYLIST_LENGTH=MIN_PLAYLIST_LENGTH,
    MAX_PLAYLISTS=MAX_PLAYLISTS,
    MAX_ITEMS=MAX_ITEMS,
    MAX_SEQ_LEN=MAX_SEQ_LEN,
))
print("Subsample:", dict(
    SUBSAMPLE_FRACTION=SUBSAMPLE_FRACTION,
    SUBSAMPLE_RANDOM_STATE=SUBSAMPLE_RANDOM_STATE,
))
print("Model:", dict(
    D_MODEL=D_MODEL,
    N_HEADS=N_HEADS,
    N_LAYERS=N_LAYERS,
    DROPOUT=DROPOUT,
    GRU_HIDDEN=GRU_HIDDEN,
    N_INTERESTS=N_INTERESTS,
    FMLP_N_LAYERS=FMLP_N_LAYERS,
))
print("Train:", dict(
    BATCH_SIZE=BATCH_SIZE,
    LR=LR,
    WEIGHT_DECAY=WEIGHT_DECAY,
    EPOCHS=EPOCHS,
    PATIENCE=PATIENCE,
))
print("SSL:", dict(
    DO_CL4SREC_SSL=DO_CL4SREC_SSL,
    DO_SIMDCL=DO_SIMDCL,
    SSL_EPOCHS=SSL_EPOCHS,
    SSL_TAU=SSL_TAU,
    SIMDCL_NOISE_STD=SIMDCL_NOISE_STD,
))
print("ComiRec:", dict(
    COMIREC_LABEL_AWARE=COMIREC_LABEL_AWARE,
    COMIREC_DIVERSITY_REG=COMIREC_DIVERSITY_REG,
    COMIREC_LABEL_TAU=COMIREC_LABEL_TAU,
))
print("Text:", dict(
    ITEM_TITLE_MAX_TOKENS=ITEM_TITLE_MAX_TOKENS,
    PLAYLIST_TITLE_MAX_TOKENS=PLAYLIST_TITLE_MAX_TOKENS,
    TEXT_VOCAB_MAX=TEXT_VOCAB_MAX,
))
print("Eval:", dict(
    TOPK=TOPK,
    EVAL_BATCH_SIZE=EVAL_BATCH_SIZE,
    ITEM_CHUNK_SIZE=ITEM_CHUNK_SIZE,
    NUM_EVAL_PLAYLISTS=NUM_EVAL_PLAYLISTS,
    SEEDS=SEEDS,
))
print("IRRT:", dict(
    IRRT_RETRIEVAL_CUTOFFS=IRRT_RETRIEVAL_CUTOFFS,
    IRRT_RERANK_CANDIDATE_SIZE=IRRT_RERANK_CANDIDATE_SIZE,
    IRRT_RERANK_TOPK=IRRT_RERANK_TOPK,
    IRRT_HISTORY_TEXT_LAST_N=IRRT_HISTORY_TEXT_LAST_N,
    IRRT_TITLE_QUERY_WEIGHT=IRRT_TITLE_QUERY_WEIGHT,
    IRRT_HISTORY_QUERY_WEIGHT=IRRT_HISTORY_QUERY_WEIGHT,
    IRRT_HYBRID_DENSE_WEIGHT=IRRT_HYBRID_DENSE_WEIGHT,
    IRRT_HYBRID_SPARSE_WEIGHT=IRRT_HYBRID_SPARSE_WEIGHT,
    EXACT_DENSE_PIPELINE_NAME=EXACT_DENSE_PIPELINE_NAME,
    HYBRID_PIPELINE_NAME=HYBRID_PIPELINE_NAME,
))
print("DEVICE:", DEVICE)

SPOTIFY_DIR: C:\Users\sokos\DataspellProjects\MML_Project\data\spotify_mpd
MODEL_DIR: C:\Users\sokos\DataspellProjects\MML_Project\saved_model_checkpoints
MPD_MAX_SLICES: None
Filtering: {'MIN_PLAYLIST_LENGTH': 10, 'MAX_PLAYLISTS': None, 'MAX_ITEMS': 20000, 'MAX_SEQ_LEN': 100}
Subsample: {'SUBSAMPLE_FRACTION': 0.7, 'SUBSAMPLE_RANDOM_STATE': 2026}
Model: {'D_MODEL': 256, 'N_HEADS': 8, 'N_LAYERS': 3, 'DROPOUT': 0.1, 'GRU_HIDDEN': 256, 'N_INTERESTS': 4, 'FMLP_N_LAYERS': 2}
Train: {'BATCH_SIZE': 1024, 'LR': 0.001, 'WEIGHT_DECAY': 0.0001, 'EPOCHS': 30, 'PATIENCE': 3}
SSL: {'DO_CL4SREC_SSL': True, 'DO_SIMDCL': True, 'SSL_EPOCHS': 10, 'SSL_TAU': 0.2, 'SIMDCL_NOISE_STD': 0.1}
ComiRec: {'COMIREC_LABEL_AWARE': True, 'COMIREC_DIVERSITY_REG': 0.05, 'COMIREC_LABEL_TAU': 1.0}
Text: {'ITEM_TITLE_MAX_TOKENS': 8, 'PLAYLIST_TITLE_MAX_TOKENS': 8, 'TEXT_VOCAB_MAX': 20000}
Eval: {'TOPK': 20, 'EVAL_BATCH_SIZE': 1024, 'ITEM_CHUNK_SIZE': 16384, 'NUM_EVAL_PLAYLISTS': None, 'SEEDS': [42, 43, 44]}
IRRT: {'IRRT_R

## 2. Dataset discovery and raw MPD loading

In this section, the notebook:
- extracts the local MPD archive,
- locates MPD slice files,
- checks that the dataset is available,
- and prepares the raw slice list for parsing.

In [3]:
def extract_local_zip(zip_path: Optional[Path], out_dir: Path) -> None:
    if zip_path is None:
        print("MPD_ZIP_PATH is None (assuming the dataset is already extracted).")
        return
    zip_path = Path(zip_path)
    if not zip_path.exists():
        raise FileNotFoundError(f"Zip archive not found: {zip_path}")
    print(f"Extracting {zip_path} -> {out_dir}")
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(out_dir)
    print("Extraction complete.")

extract_local_zip(MPD_ZIP_PATH, SPOTIFY_DIR)

slice_files = sorted(SPOTIFY_DIR.rglob("mpd.slice.*.json"))
if MPD_MAX_SLICES is not None:
    slice_files = slice_files[:MPD_MAX_SLICES]

print("Found slice files:", len(slice_files))
if len(slice_files) == 0:
    raise FileNotFoundError(
        "No MPD slice files were found. Put extracted files into data/spotify_mpd/ "
        "so that files like 'mpd.slice.0-999.json' are visible."
    )
print("First slice:", slice_files[0])
if len(slice_files) > 1:
    print("Last slice:", slice_files[-1])

MPD_ZIP_PATH is None (assuming the dataset is already extracted).
Found slice files: 1000
First slice: C:\Users\sokos\DataspellProjects\MML_Project\data\spotify_mpd\mpd.slice.0-999.json
Last slice: C:\Users\sokos\DataspellProjects\MML_Project\data\spotify_mpd\mpd.slice.999000-999999.json


## 3. Parse playlist-track interactions

Here I convert MPD slices into a simple interaction table with:
- playlist ID,
- track ID,
- and track position inside the playlist.

This is the raw sequential interaction format used by the rest of the pipeline.

In [4]:
# def safe_str(x) -> str:
#     return "" if x is None else str(x)


# def duration_bucket(ms: int) -> str:
#     if ms <= 0:
#         return "unk"
#     sec = ms / 1000.0
#     if sec < 60:
#         return "<1m"
#     if sec < 120:
#         return "1-2m"
#     if sec < 180:
#         return "2-3m"
#     if sec < 240:
#         return "3-4m"
#     if sec < 300:
#         return "4-5m"
#     return "5m+"


# def parse_mpd_slices_to_sequences(files: List[Path]) -> pd.DataFrame:
#     rows = []
#     for path in files:
#         with open(path, "r", encoding="utf-8") as f:
#             data = json.load(f)

#         for pl in data.get("playlists", []):
#             pid = int(pl["pid"])
#             playlist_name = safe_str(pl.get("name")).strip().lower()

#             for pos, tr in enumerate(pl.get("tracks", [])):
#                 track_uri = tr.get("track_uri")
#                 if track_uri is None:
#                     continue

#                 duration_ms = int(tr.get("duration_ms", 0) or 0)

#                 rows.append({
#                     "pid": pid,
#                     "playlist_name": playlist_name,
#                     "track_uri": safe_str(track_uri),
#                     "track_name": safe_str(tr.get("track_name")).strip().lower(),
#                     "artist_name": safe_str(tr.get("artist_name")).strip().lower(),
#                     "album_name": safe_str(tr.get("album_name")).strip().lower(),
#                     "artist_uri": safe_str(tr.get("artist_uri")),
#                     "album_uri": safe_str(tr.get("album_uri")),
#                     "duration_ms": duration_ms,
#                     "duration_bucket": duration_bucket(duration_ms),
#                     "pos": int(pos),
#                 })
#     return pd.DataFrame(rows)

# interactions = parse_mpd_slices_to_sequences(slice_files)
# print("Parsed interactions:", interactions.shape)
# display(interactions.head())

## 4. Interaction cleaning and dataset filtering

This stage:
- sorts interactions by playlist and position,
- removes exact duplicates,
- filters out very short playlists,
- optionally caps the number of playlists and items,
- and applies a stratified subsample by playlist length.

In [5]:
# interactions = interactions.copy()
# interactions["pid"] = interactions["pid"].astype(int)
# interactions["track_uri"] = interactions["track_uri"].astype(str)
# interactions["playlist_name"] = interactions["playlist_name"].fillna("").astype(str)
# interactions["track_name"] = interactions["track_name"].fillna("").astype(str)
# interactions["artist_name"] = interactions["artist_name"].fillna("").astype(str)
# interactions["album_name"] = interactions["album_name"].fillna("").astype(str)
# interactions["artist_uri"] = interactions["artist_uri"].fillna("").astype(str)
# interactions["album_uri"] = interactions["album_uri"].fillna("").astype(str)
# interactions["duration_ms"] = pd.to_numeric(interactions["duration_ms"], errors="coerce").fillna(0).astype(int)
# interactions["duration_bucket"] = interactions["duration_bucket"].fillna("unk").astype(str)
# interactions["pos"] = interactions["pos"].astype(int)

# interactions = (
#     interactions
#     .sort_values(["pid", "pos"])
#     .drop_duplicates(subset=["pid", "track_uri", "pos"])
#     .reset_index(drop=True)
# )
# print("Cleaned interactions:", interactions.shape)
# display(interactions.head())

In [6]:
# interactions.to_parquet(SAVE_DIR / "interactions.parquet", index=False)

In [7]:
# interactions = pd.read_parquet(SAVE_DIR / "interactions.parquet")
# print("Reloaded interactions:", interactions.shape)

In [8]:
# def filter_playlists_by_min_length(df: pd.DataFrame, min_len: int) -> pd.DataFrame:
#     cnt = df.groupby("pid").size()
#     keep = cnt[cnt >= min_len].index
#     return df[df["pid"].isin(keep)].copy()


# def cap_top_playlists(df: pd.DataFrame, max_playlists: Optional[int]) -> pd.DataFrame:
#     if max_playlists is None or df["pid"].nunique() <= max_playlists:
#         return df
#     top_playlists = (
#         df.groupby("pid")
#         .size()
#         .sort_values(ascending=False)
#         .head(max_playlists)
#         .index
#     )
#     return df[df["pid"].isin(top_playlists)].copy()


# def cap_top_items(df: pd.DataFrame, max_items: Optional[int]) -> pd.DataFrame:
#     if max_items is None or df["track_uri"].nunique() <= max_items:
#         return df
#     top_items = (
#         df.groupby("track_uri")
#         .size()
#         .sort_values(ascending=False)
#         .head(max_items)
#         .index
#     )
#     return df[df["track_uri"].isin(top_items)].copy()


# def build_playlist_meta(df: pd.DataFrame) -> pd.DataFrame:
#     meta = (
#         df.groupby("pid")
#         .size()
#         .rename("playlist_len")
#         .reset_index()
#     )

#     bins = [9, 19, 39, 79, 199, np.inf]
#     labels = ["10-19", "20-39", "40-79", "80-199", "200+"]

#     meta["len_bin"] = pd.cut(
#         meta["playlist_len"],
#         bins=bins,
#         labels=labels,
#         right=True,
#     )
#     return meta


# def add_length_bin(df: pd.DataFrame) -> pd.DataFrame:
#     meta = build_playlist_meta(df)
#     out = df.merge(meta, on="pid", how="left")
#     return out


# def stratified_playlist_subsample(
#     df: pd.DataFrame,
#     fraction: float,
#     random_state: int,
# ) -> pd.DataFrame:
#     meta = build_playlist_meta(df)

#     sampled_pids: List[int] = []
#     rng = np.random.RandomState(random_state)

#     for _, g in meta.groupby("len_bin", observed=False):
#         if len(g) == 0:
#             continue

#         n_take = max(1, int(round(len(g) * fraction)))
#         n_take = min(n_take, len(g))

#         sampled = g.sample(
#             n=n_take,
#             random_state=int(rng.randint(0, 10**9)),
#         )["pid"].tolist()

#         sampled_pids.extend(sampled)

#     sampled_pids = sorted(set(sampled_pids))
#     return df[df["pid"].isin(sampled_pids)].copy()

In [9]:
# df = interactions.copy()
# df = filter_playlists_by_min_length(df, MIN_PLAYLIST_LENGTH)
# df = cap_top_playlists(df, MAX_PLAYLISTS)
# df = cap_top_items(df, MAX_ITEMS)
# df = filter_playlists_by_min_length(df, MIN_PLAYLIST_LENGTH)

# df = stratified_playlist_subsample(df, SUBSAMPLE_FRACTION, SUBSAMPLE_RANDOM_STATE)
# df = filter_playlists_by_min_length(df, MIN_PLAYLIST_LENGTH)
# df = add_length_bin(df)
# df = df.sort_values(["pid", "pos"]).reset_index(drop=True)

# print("Interactions after stratified subsample:", df.shape)
# print("Unique playlists after subsample:", df["pid"].nunique())
# print("Unique tracks after subsample:", df["track_uri"].nunique())

In [10]:
# df.to_parquet(SAVE_DIR / f"interactions_subsample_{SUBSAMPLE_FRACTION}_maxitems_{MAX_ITEMS}.parquet", index=False)

In [11]:
df = pd.read_parquet(SAVE_DIR / f"interactions_subsample_{SUBSAMPLE_FRACTION}_maxitems_{MAX_ITEMS}.parquet")
print("Reloaded subsampled interactions:", df.shape)

Reloaded subsampled interactions: (29655355, 13)


## 5. Remap playlists and tracks to integer IDs

The models operate on compact integer IDs rather than raw playlist or track identifiers.

This section:
- maps playlists to user IDs,
- maps tracks to item IDs,
- and prepares the final interaction table used for sequence modeling.

In [12]:
def remap_series_to_int(s: pd.Series) -> Tuple[pd.Series, Dict[str, int]]:
    uniq = s.unique()
    mapping = {v: i + 1 for i, v in enumerate(uniq)}
    return s.map(mapping).astype(np.int32), mapping


df["user_id"], playlist_map = remap_series_to_int(df["pid"].astype(str))
df["item_id"], item_map = remap_series_to_int(df["track_uri"])
df = df.drop(columns=["track_uri"])

n_users = int(df["user_id"].max())
n_items = int(df["item_id"].max())

print("Filtered interactions:", df.shape)
print("Playlists:", n_users, "| Tracks:", n_items)
display(df.head())

Filtered interactions: (29655355, 14)
Playlists: 572623 | Tracks: 20000


,pid,playlist_name,track_name,artist_name,album_name,artist_uri,album_uri,duration_ms,duration_bucket,pos,playlist_len,len_bin,user_id,item_id
0,3,mat,when did your heart go missing?,rooney,calling the world,spotify:artist:78inRkIIF0n9R9I2HPoWP7,spotify:album:7kHRGtUgh8UTZhN3A8eJF1,212926,3-4m,23,26,20-39,1,1
1,3,mat,two weeks,grizzly bear,veckatimest,spotify:artist:2Jv5eshHtLycR6R8KQCdc4,spotify:album:6FIFqclBriPCb0SjWDaHIk,243160,4-5m,26,26,20-39,1,2
2,3,mat,yet again,grizzly bear,shields,spotify:artist:2Jv5eshHtLycR6R8KQCdc4,spotify:album:57LAEzKL94ZHwbIkUWYCDY,318423,5m+,27,26,20-39,1,3
3,3,mat,campus,vampire weekend,vampire weekend,spotify:artist:5BvJzeQpmsdsFp4HGUYUEx,spotify:album:5oXBmKbyJeQftWMo87cQ9F,176466,2-3m,28,26,20-39,1,4
4,3,mat,talk to me,kopecky,drug for the modern age,spotify:artist:0vRO9bFgOrDoFtLcDHV8b6,spotify:album:581UyGGQZpWEFMdv7BLlQA,173826,2-3m,30,26,20-39,1,5


In [13]:
item_freq = df["item_id"].value_counts()
print("Most frequent item_id:", int(item_freq.index[0]), "count:", int(item_freq.iloc[0]))
print("Unique items:", df["item_id"].nunique())

Most frequent item_id: 458 count: 32249
Unique items: 20000


## 5.1 Build item and playlist metadata tables

In this section, I derive compact metadata tables that can be used for:
- metadata-aware slicing,
- music-aware diversity analysis,
- artist / album based sequence features,
- and later metadata-aware model extensions.

In [14]:
def tokenize_text_simple(text: str) -> List[str]:
    text = (text or "").strip().lower()
    if not text:
        return []
    return [tok for tok in text.replace("/", " ").replace("-", " ").split() if tok]


def build_token_vocab(
    texts: List[str],
    min_freq: int = 1,
    max_vocab: Optional[int] = 20000,
) -> Dict[str, int]:
    cnt = Counter()
    for text in texts:
        cnt.update(tokenize_text_simple(text))

    items = [(tok, freq) for tok, freq in cnt.items() if freq >= min_freq]
    items.sort(key=lambda x: (-x[1], x[0]))

    if max_vocab is not None:
        items = items[:max_vocab]

    vocab = {"<pad>": 0, "<unk>": 1}
    for tok, _ in items:
        vocab[tok] = len(vocab)
    return vocab


def encode_text_tokens(text: str, vocab: Dict[str, int], max_len: int) -> List[int]:
    toks = tokenize_text_simple(text)
    ids = [vocab.get(tok, vocab["<unk>"]) for tok in toks[:max_len]]
    if len(ids) < max_len:
        ids = ids + [vocab["<pad>"]] * (max_len - len(ids))
    return ids


def remap_with_pad(values: pd.Series) -> Tuple[pd.Series, Dict[str, int]]:
    uniq = sorted(values.fillna("").astype(str).unique().tolist())
    mapping = {"": 0}
    for v in uniq:
        if v == "":
            continue
        mapping[v] = len(mapping)
    return values.fillna("").astype(str).map(mapping).astype(np.int32), mapping


@dataclass
class SeqData:
    user_seqs: Dict[int, List[int]]
    n_users: int
    n_items: int


def right_pad(seq, max_len: int, pad=0):
    if len(seq) >= max_len:
        return list(seq[-max_len:])
    return list(seq) + [pad for _ in range(max_len - len(seq))]


def split_train_valid_test(seqdata: SeqData):
    train_seqs, valid_seqs, test_seqs = {}, {}, {}
    for u, seq in seqdata.user_seqs.items():
        if len(seq) < 3:
            continue
        train_seqs[u] = seq[:-2]
        valid_seqs[u] = seq[:-1]
        test_seqs[u] = seq[:]
    return train_seqs, valid_seqs, test_seqs


def set_all_seeds(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def build_fixed_eval_users(
    eval_seqs: Dict[int, List[int]],
    num_eval_users: Optional[int],
    seed: int,
) -> List[int]:
    users = list(eval_seqs.keys())
    rng = random.Random(seed)
    rng.shuffle(users)
    if num_eval_users is None:
        return users
    return users[:min(num_eval_users, len(users))]


@dataclass
class EvalCases:
    users: List[int]
    histories: np.ndarray
    targets: np.ndarray


def build_eval_cases(
    eval_seqs: Dict[int, List[int]],
    users: List[int],
    max_len: int,
) -> EvalCases:
    histories, targets = [], []

    for u in users:
        seq = eval_seqs[u]
        hist = right_pad(seq[:-1], max_len)
        target = seq[-1]

        histories.append(hist)
        targets.append(target)

    return EvalCases(
        users=users,
        histories=np.asarray(histories, dtype=np.int64),
        targets=np.asarray(targets, dtype=np.int64),
    )


def filter_eval_cases_no_repeat(
    eval_cases: EvalCases,
    eval_seqs: Dict[int, List[int]],
) -> EvalCases:
    keep_users = []
    keep_histories = []
    keep_targets = []

    for u, hist, target in zip(eval_cases.users, eval_cases.histories, eval_cases.targets):
        full_seq = eval_seqs[u]
        prefix = full_seq[:-1]
        if int(target) in set(prefix):
            continue

        keep_users.append(u)
        keep_histories.append(hist)
        keep_targets.append(target)

    return EvalCases(
        users=keep_users,
        histories=np.asarray(keep_histories, dtype=np.int64),
        targets=np.asarray(keep_targets, dtype=np.int64),
    )


# build sequences first
user_seqs = df.groupby("user_id")["item_id"].apply(list).to_dict()
seqdata = SeqData(user_seqs=user_seqs, n_users=n_users, n_items=n_items)

train_seqs, valid_seqs, test_seqs = split_train_valid_test(seqdata)
print("Playlists with train/valid/test sequences:", len(train_seqs))

# build full metadata tables
item_meta = (
    df.groupby("item_id", as_index=False)
    .agg(
        track_name=("track_name", "first"),
        artist_name=("artist_name", "first"),
        album_name=("album_name", "first"),
        artist_uri=("artist_uri", "first"),
        album_uri=("album_uri", "first"),
        duration_bucket=("duration_bucket", "first"),
    )
)

playlist_meta = (
    df.groupby("user_id", as_index=False)
    .agg(
        playlist_name=("playlist_name", "first"),
    )
)

item_meta["artist_id"], artist_map = remap_with_pad(item_meta["artist_uri"])
item_meta["album_id"], album_map = remap_with_pad(item_meta["album_uri"])
item_meta["duration_id"], duration_map = remap_with_pad(item_meta["duration_bucket"])

# train-only vocab to reduce leakage
train_user_ids = sorted(train_seqs.keys())
train_item_ids = sorted({x for seq in train_seqs.values() for x in seq})

track_title_vocab = build_token_vocab(
    item_meta.loc[item_meta["item_id"].isin(train_item_ids), "track_name"]
    .fillna("")
    .astype(str)
    .tolist(),
    max_vocab=TEXT_VOCAB_MAX,
)

playlist_title_vocab = build_token_vocab(
    playlist_meta.loc[playlist_meta["user_id"].isin(train_user_ids), "playlist_name"]
    .fillna("")
    .astype(str)
    .tolist(),
    max_vocab=TEXT_VOCAB_MAX,
)

item_meta["track_title_token_ids"] = item_meta["track_name"].map(
    lambda x: encode_text_tokens(x, track_title_vocab, ITEM_TITLE_MAX_TOKENS)
)

playlist_meta["playlist_title_token_ids"] = playlist_meta["playlist_name"].map(
    lambda x: encode_text_tokens(x, playlist_title_vocab, PLAYLIST_TITLE_MAX_TOKENS)
)

item_feature_table = item_meta[
    [
        "item_id",
        "artist_id",
        "album_id",
        "duration_id",
        "track_title_token_ids",
        "track_name",
        "artist_name",
        "album_name",
    ]
].copy()

playlist_feature_table = playlist_meta[
    [
        "user_id",
        "playlist_title_token_ids",
        "playlist_name",
    ]
].copy()

itemid_to_artist_id = dict(zip(item_feature_table["item_id"], item_feature_table["artist_id"]))
itemid_to_album_id = dict(zip(item_feature_table["item_id"], item_feature_table["album_id"]))
itemid_to_duration_id = dict(zip(item_feature_table["item_id"], item_feature_table["duration_id"]))
itemid_to_track_title_tokens = dict(zip(item_feature_table["item_id"], item_feature_table["track_title_token_ids"]))

userid_to_playlist_title_tokens = dict(
    zip(playlist_feature_table["user_id"], playlist_feature_table["playlist_title_token_ids"])
)

itemid_to_artist = dict(zip(item_meta["item_id"], item_meta["artist_uri"]))
itemid_to_album = dict(zip(item_meta["item_id"], item_meta["album_uri"]))
itemid_to_duration_bucket = dict(zip(item_meta["item_id"], item_meta["duration_bucket"]))
itemid_to_track_name = dict(zip(item_meta["item_id"], item_meta["track_name"]))
itemid_to_artist_name = dict(zip(item_meta["item_id"], item_meta["artist_name"]))
itemid_to_album_name = dict(zip(item_meta["item_id"], item_meta["album_name"]))
userid_to_playlist_title = dict(zip(playlist_meta["user_id"], playlist_meta["playlist_name"]))

N_ARTISTS = int(item_meta["artist_id"].max()) + 1
N_ALBUMS = int(item_meta["album_id"].max()) + 1
N_DURATIONS = int(item_meta["duration_id"].max()) + 1
N_TRACK_TITLE_TOKENS = len(track_title_vocab)
N_PLAYLIST_TITLE_TOKENS = len(playlist_title_vocab)

display(item_feature_table.head())
display(playlist_feature_table.head())

print("Artists vocab:", N_ARTISTS)
print("Albums vocab:", N_ALBUMS)
print("Duration vocab:", N_DURATIONS)
print("Track title vocab:", N_TRACK_TITLE_TOKENS)
print("Playlist title vocab:", N_PLAYLIST_TITLE_TOKENS)

# freeze eval users, then filter unreachable repeated targets
EVAL_USER_SEED = 2026

valid_users_fixed = build_fixed_eval_users(valid_seqs, NUM_EVAL_PLAYLISTS, EVAL_USER_SEED)
test_users_fixed = build_fixed_eval_users(test_seqs, NUM_EVAL_PLAYLISTS, EVAL_USER_SEED)

valid_cases_full = build_eval_cases(valid_seqs, valid_users_fixed, MAX_SEQ_LEN)
test_cases_full = build_eval_cases(test_seqs, test_users_fixed, MAX_SEQ_LEN)

valid_cases = filter_eval_cases_no_repeat(valid_cases_full, valid_seqs)
test_cases = filter_eval_cases_no_repeat(test_cases_full, test_seqs)

print("Validation cases before no-repeat filter:", len(valid_cases_full.users))
print("Validation cases after no-repeat filter:", len(valid_cases.users))
print("Test cases before no-repeat filter:", len(test_cases_full.users))
print("Test cases after no-repeat filter:", len(test_cases.users))

Playlists with train/valid/test sequences: 572623


,item_id,artist_id,album_id,duration_id,track_title_token_ids,track_name,artist_name,album_name
0,1,4276,10454,3,"[77, 425, 21, 52, 40, 8844, 0, 0]",when did your heart go missing?,rooney,calling the world
1,2,1412,8610,4,"[217, 1725, 0, 0, 0, 0, 0, 0]",two weeks,grizzly bear,veckatimest
2,3,1412,7015,5,"[1080, 131, 0, 0, 0, 0, 0, 0]",yet again,grizzly bear,shields
3,4,3102,7996,2,"[6143, 0, 0, 0, 0, 0, 0, 0]",campus,vampire weekend,vampire weekend
4,5,567,7033,2,"[237, 13, 6, 0, 0, 0, 0, 0]",talk to me,kopecky,drug for the modern age


,user_id,playlist_title_token_ids,playlist_name
0,1,"[6376, 0, 0, 0, 0, 0, 0, 0]",mat
1,2,"[65, 0, 0, 0, 0, 0, 0, 0]",90s
2,3,"[33, 0, 0, 0, 0, 0, 0, 0]",wedding
3,4,"[808, 0, 0, 0, 0, 0, 0, 0]",bop
4,5,"[1006, 0, 0, 0, 0, 0, 0, 0]",abby


Artists vocab: 4671
Albums vocab: 10819
Duration vocab: 7
Track title vocab: 11662
Playlist title vocab: 17819
Validation cases before no-repeat filter: 572623
Validation cases after no-repeat filter: 559314
Test cases before no-repeat filter: 572623
Test cases after no-repeat filter: 558270


In [15]:
ITEM_ARTIST_IDS_CPU = torch.zeros(n_items + 1, dtype=torch.long)
ITEM_ALBUM_IDS_CPU = torch.zeros(n_items + 1, dtype=torch.long)
ITEM_DURATION_IDS_CPU = torch.zeros(n_items + 1, dtype=torch.long)
ITEM_TRACK_TITLE_TOKENS_CPU = torch.zeros(n_items + 1, ITEM_TITLE_MAX_TOKENS, dtype=torch.long)

for item_id in range(1, n_items + 1):
    ITEM_ARTIST_IDS_CPU[item_id] = itemid_to_artist_id.get(item_id, 0)
    ITEM_ALBUM_IDS_CPU[item_id] = itemid_to_album_id.get(item_id, 0)
    ITEM_DURATION_IDS_CPU[item_id] = itemid_to_duration_id.get(item_id, 0)
    ITEM_TRACK_TITLE_TOKENS_CPU[item_id] = torch.tensor(
        itemid_to_track_title_tokens.get(item_id, [0] * ITEM_TITLE_MAX_TOKENS),
        dtype=torch.long,
    )

USER_PLAYLIST_TITLE_TOKENS_CPU = torch.zeros(n_users + 1, PLAYLIST_TITLE_MAX_TOKENS, dtype=torch.long)
for user_id in range(1, n_users + 1):
    USER_PLAYLIST_TITLE_TOKENS_CPU[user_id] = torch.tensor(
        userid_to_playlist_title_tokens.get(user_id, [0] * PLAYLIST_TITLE_MAX_TOKENS),
        dtype=torch.long,
    )

if torch.cuda.is_available():
    ITEM_ARTIST_IDS_GPU = ITEM_ARTIST_IDS_CPU.to(DEVICE, non_blocking=True)
    ITEM_ALBUM_IDS_GPU = ITEM_ALBUM_IDS_CPU.to(DEVICE, non_blocking=True)
    ITEM_DURATION_IDS_GPU = ITEM_DURATION_IDS_CPU.to(DEVICE, non_blocking=True)
    ITEM_TRACK_TITLE_TOKENS_GPU = ITEM_TRACK_TITLE_TOKENS_CPU.to(DEVICE, non_blocking=True)
    USER_PLAYLIST_TITLE_TOKENS_GPU = USER_PLAYLIST_TITLE_TOKENS_CPU.to(DEVICE, non_blocking=True)
else:
    ITEM_ARTIST_IDS_GPU = ITEM_ARTIST_IDS_CPU
    ITEM_ALBUM_IDS_GPU = ITEM_ALBUM_IDS_CPU
    ITEM_DURATION_IDS_GPU = ITEM_DURATION_IDS_CPU
    ITEM_TRACK_TITLE_TOKENS_GPU = ITEM_TRACK_TITLE_TOKENS_CPU
    USER_PLAYLIST_TITLE_TOKENS_GPU = USER_PLAYLIST_TITLE_TOKENS_CPU

## 5.2 Build lexical retrieval features for IRRT

This section builds sparse lexical representations for:
- playlist retrieval queries
- item retrieval documents

These features are used in the IRRT experiments for:
- sparse lexical retrieval
- hybrid sparse+dense retrieval
- retrieval candidate recall evaluation
- shortlist reranking evaluation

In [16]:
def build_item_lexical_text(row: pd.Series) -> str:
    parts = [
        str(row.get("track_name", "")),
        str(row.get("artist_name", "")),
        str(row.get("album_name", "")),
    ]
    return " ".join([p.strip().lower() for p in parts if str(p).strip()])


item_meta["lexical_text"] = item_meta.apply(build_item_lexical_text, axis=1)
playlist_meta["playlist_title_text"] = playlist_meta["playlist_name"].fillna("").astype(str).str.lower()

itemid_to_lexical_text = dict(zip(item_meta["item_id"], item_meta["lexical_text"]))
userid_to_playlist_title_text = dict(zip(playlist_meta["user_id"], playlist_meta["playlist_title_text"]))


def build_doc_freq(texts: List[str]) -> Counter:
    df_counter = Counter()
    for text in texts:
        toks = set(tokenize_text_simple(text))
        df_counter.update(toks)
    return df_counter


def text_to_tf_counter(text: str) -> Dict[str, int]:
    toks = tokenize_text_simple(text)
    return dict(Counter(toks))


def bm25_idf(df_val: int, n_docs: int) -> float:
    return math.log(1.0 + (n_docs - df_val + 0.5) / (df_val + 0.5))


BM25_K1 = 1.2
BM25_B = 0.75

LEXICAL_DF = build_doc_freq(item_meta["lexical_text"].tolist())
LEXICAL_N_DOCS = max(1, len(item_meta))

itemid_to_tf_vec = {
    item_id: text_to_tf_counter(itemid_to_lexical_text.get(item_id, ""))
    for item_id in range(1, n_items + 1)
}

itemid_to_doc_len = {
    item_id: int(sum(tf_vec.values()))
    for item_id, tf_vec in itemid_to_tf_vec.items()
}

BM25_AVGDL = float(np.mean(list(itemid_to_doc_len.values()))) if itemid_to_doc_len else 1.0

print("Lexical sparse representations prepared.")
print("Example item lexical text:", item_meta.loc[0, "lexical_text"] if len(item_meta) > 0 else "")
print("BM25 avgdl:", BM25_AVGDL)

Lexical sparse representations prepared.
Example item lexical text: when did your heart go missing? rooney calling the world
BM25 avgdl: 8.14625


## 6. Build sequential train/validation/test splits

For each playlist, I build an ordered item sequence and use a simple last-item protocol:
- train sequence: all items except the last two,
- validation sequence: all items except the last one,
- test sequence: full sequence with the last item as the target.

This keeps the setup consistent with next-item prediction.

## 7. Freeze evaluation users and evaluation cases

To make model comparison fair, I freeze:
- validation playlists,
- test playlists,
- and the exact sequence histories used during evaluation.

This ensures that all models are scored on the same cases.

## 8. Build analysis metadata and subset definitions

Besides global test metrics, the notebook evaluates targeted subsets that are consistent with the no-repeat playlist continuation setting.

This section creates:
- length-based slices,
- metadata-aware heterogeneity slices,
- target popularity slices,
- unseen-target slices,
- and hard-continuation slices.

In [17]:
def safe_norm_entropy_from_counts(counts: np.ndarray) -> float:
    counts = counts[counts > 0].astype(np.float64)
    if len(counts) <= 1:
        return 0.0
    p = counts / counts.sum()
    ent = -(p * np.log(p + 1e-12)).sum()
    max_ent = math.log(len(p))
    return 0.0 if max_ent <= 0 else float(ent / max_ent)


def compute_history_diversity_features(
    seq: List[int],
    artist_map: Dict[int, str],
    album_map: Dict[int, str],
    duration_bucket_map: Dict[int, str],
) -> Dict[str, float]:
    hist = list(seq)
    L = len(hist)

    if L == 0:
        return {
            "hist_len": 0.0,
            "item_unique_ratio": 0.0,
            "artist_unique_ratio": 0.0,
            "album_unique_ratio": 0.0,
            "item_entropy": 0.0,
            "artist_entropy": 0.0,
            "duration_entropy": 0.0,
            "diversity_score": 0.0,
        }

    item_counts = np.array(list(Counter(hist).values()), dtype=np.int64)

    artists = [artist_map.get(x, "") for x in hist]
    albums = [album_map.get(x, "") for x in hist]
    durations = [duration_bucket_map.get(x, "unk") for x in hist]

    artist_counts = np.array(list(Counter(artists).values()), dtype=np.int64)
    duration_counts = np.array(list(Counter(durations).values()), dtype=np.int64)

    item_unique_ratio = len(set(hist)) / L
    artist_unique_ratio = len(set(artists)) / L
    album_unique_ratio = len(set(albums)) / L

    item_entropy = safe_norm_entropy_from_counts(item_counts)
    artist_entropy = safe_norm_entropy_from_counts(artist_counts)
    duration_entropy = safe_norm_entropy_from_counts(duration_counts)

    diversity_score = float(
        0.35 * item_unique_ratio
        + 0.35 * artist_entropy
        + 0.15 * album_unique_ratio
        + 0.15 * duration_entropy
    )

    return {
        "hist_len": float(L),
        "item_unique_ratio": float(item_unique_ratio),
        "artist_unique_ratio": float(artist_unique_ratio),
        "album_unique_ratio": float(album_unique_ratio),
        "item_entropy": float(item_entropy),
        "artist_entropy": float(artist_entropy),
        "duration_entropy": float(duration_entropy),
        "diversity_score": diversity_score,
    }


def build_eval_analysis_meta(
    eval_cases: EvalCases,
    eval_seqs: Dict[int, List[int]],
    track_pop: pd.Series,
    artist_map: Dict[int, str],
    album_map: Dict[int, str],
    duration_bucket_map: Dict[int, str],
    hard_window: int = 5,
) -> pd.DataFrame:
    rows = []

    pop_values = track_pop.astype(float)
    tail_thr = float(pop_values.quantile(0.20))
    head_thr = float(pop_values.quantile(0.80))

    for idx, u in enumerate(eval_cases.users):
        full_seq = eval_seqs[u]
        hist = full_seq[:-1]
        target = full_seq[-1]

        f = compute_history_diversity_features(
            seq=hist,
            artist_map=artist_map,
            album_map=album_map,
            duration_bucket_map=duration_bucket_map,
        )

        target_pop = float(track_pop.get(target, 0.0))
        if target_pop <= tail_thr:
            target_pop_bucket = "tail"
        elif target_pop >= head_thr:
            target_pop_bucket = "head"
        else:
            target_pop_bucket = "mid"

        last_h = hist[-hard_window:] if len(hist) >= hard_window else hist[:]
        hard_continuation = int(target not in last_h)

        rows.append(
            {
                "row_idx": idx,
                "user_id": u,
                "target": target,
                "target_pop": target_pop,
                "target_pop_bucket": target_pop_bucket,
                "hard_continuation": hard_continuation,
                **f,
            }
        )

    meta = pd.DataFrame(rows)

    q_len_33 = meta["hist_len"].quantile(0.33)
    q_len_67 = meta["hist_len"].quantile(0.67)

    def len_bucket_fn(x: float) -> str:
        if x <= q_len_33:
            return "short"
        if x >= q_len_67:
            return "long"
        return "medium"

    meta["len_bucket"] = meta["hist_len"].map(len_bucket_fn)

    q_div_33 = meta["diversity_score"].quantile(0.33)
    q_div_67 = meta["diversity_score"].quantile(0.67)

    def div_bucket_fn(x: float) -> str:
        if x <= q_div_33:
            return "homogeneous"
        if x >= q_div_67:
            return "heterogeneous"
        return "medium"

    meta["div_bucket"] = meta["diversity_score"].map(div_bucket_fn)
    return meta

def build_subset_masks(eval_meta: pd.DataFrame) -> Dict[str, np.ndarray]:
    masks: Dict[str, np.ndarray] = {}
    masks["all"] = np.ones(len(eval_meta), dtype=bool)

    for bucket in ["short", "medium", "long"]:
        mask = eval_meta["len_bucket"].values == bucket
        if mask.sum() > 0:
            masks[f"len_{bucket}"] = mask

    for bucket in ["homogeneous", "medium", "heterogeneous"]:
        mask = eval_meta["div_bucket"].values == bucket
        if mask.sum() > 0:
            masks[f"div_{bucket}"] = mask

    for bucket in ["head", "mid", "tail"]:
        mask = eval_meta["target_pop_bucket"].values == bucket
        if mask.sum() > 0:
            masks[f"target_pop_{bucket}"] = mask

    hard_mask = eval_meta["hard_continuation"].values.astype(bool)
    if hard_mask.sum() > 0 and hard_mask.sum() < len(eval_meta):
        masks["hard_continuation"] = hard_mask

    return masks


def metrics_at_k(ranked: np.ndarray, target: np.ndarray, k: int):
    hit = ranked[:, :k] == target[:, None]
    recall = float(hit.any(axis=1).mean())

    first_pos = hit.argmax(axis=1)
    has_hit = hit.any(axis=1)

    rr = np.zeros(hit.shape[0], dtype=np.float64)
    rr[has_hit] = 1.0 / (first_pos[has_hit] + 1.0)
    mrr = float(rr.mean())

    ndcg = np.zeros(hit.shape[0], dtype=np.float64)
    ndcg[has_hit] = 1.0 / np.log2(first_pos[has_hit] + 2.0)
    ndcg = float(ndcg.mean())

    return recall, ndcg, mrr


def coverage_at_k(ranked: np.ndarray, n_items: int, k: int) -> float:
    vals = ranked[:, :k].reshape(-1)
    vals = vals[vals > 0]
    if len(vals) == 0:
        return 0.0
    return float(len(np.unique(vals)) / n_items)


def compute_metrics_from_ranked(
    ranked: np.ndarray,
    targets: np.ndarray,
    n_items: int,
    k: int,
) -> Dict[str, float]:
    r, n, m = metrics_at_k(ranked, targets, k)
    cov = coverage_at_k(ranked, n_items, k)
    return {
        f"Recall@{k}": float(r),
        f"NDCG@{k}": float(n),
        f"MRR@{k}": float(m),
        f"Coverage@{k}": float(cov),
    }


def summarize_subset_metrics_from_ranked(
    ranked: np.ndarray,
    targets: np.ndarray,
    n_items: int,
    k: int,
    subset_masks: Dict[str, np.ndarray],
    model_name: str,
    seed,
) -> List[Dict[str, object]]:
    rows = []
    for subset_name, mask in subset_masks.items():
        ranked_sub = ranked[mask]
        targets_sub = targets[mask]
        if len(targets_sub) == 0:
            continue

        metrics = compute_metrics_from_ranked(
            ranked=ranked_sub,
            targets=targets_sub,
            n_items=n_items,
            k=k,
        )
        rows.append(
            {
                "model": model_name,
                "seed": seed,
                "subset": subset_name,
                "n_cases": int(mask.sum()),
                **metrics,
            }
        )
    return rows


train_pop_counter = Counter()
for seq in train_seqs.values():
    train_pop_counter.update(seq)

track_pop = pd.Series(
    {item_id: train_pop_counter.get(item_id, 0) for item_id in range(1, n_items + 1)},
    name="track_pop",
    dtype=np.int64,
)

test_analysis_meta = build_eval_analysis_meta(
    eval_cases=test_cases,
    eval_seqs=test_seqs,
    track_pop=track_pop,
    artist_map=itemid_to_artist,
    album_map=itemid_to_album,
    duration_bucket_map=itemid_to_duration_bucket,
)
TEST_SUBSET_MASKS = build_subset_masks(test_analysis_meta)

print("Test analysis meta shape:", test_analysis_meta.shape)
display(test_analysis_meta.head())

Test analysis meta shape: (558270, 16)


,row_idx,user_id,target,target_pop,target_pop_bucket,hard_continuation,hist_len,item_unique_ratio,artist_unique_ratio,album_unique_ratio,item_entropy,artist_entropy,duration_entropy,diversity_score,len_bucket,div_bucket
0,0,445098,18838,374.0,tail,1,121.0,1.0,0.768595,0.917355,1.0,0.966222,0.600764,0.915895,long,medium
1,1,507710,9058,8206.0,head,1,15.0,1.0,0.933333,1.000000,1.0,0.991123,0.353359,0.899897,short,homogeneous
2,2,156316,8549,4623.0,head,1,10.0,1.0,0.700000,0.700000,1.0,0.942681,0.785475,0.902760,short,homogeneous
3,3,163991,9664,758.0,mid,1,84.0,1.0,0.630952,0.880952,1.0,0.950660,0.665039,0.914630,long,medium
4,4,235025,8618,5383.0,head,1,14.0,1.0,0.857143,0.928571,1.0,0.967296,0.755928,0.941229,short,heterogeneous


In [18]:
print("Available subsets:")
for name, mask in TEST_SUBSET_MASKS.items():
    print(f"{name:>20s}: n={int(mask.sum())}")

Available subsets:
                 all: n=558270
           len_short: n=188849
          len_medium: n=182053
            len_long: n=187368
     div_homogeneous: n=184229
          div_medium: n=189812
   div_heterogeneous: n=184229
     target_pop_head: n=299382
      target_pop_mid: n=213442
     target_pop_tail: n=45446


In [19]:
pop = Counter()
for seq in train_seqs.values():
    pop.update(seq)

print("TopPop head:", [item for item, _ in pop.most_common(TOPK)])

TopPop head: [458, 1840, 510, 632, 1268, 1251, 456, 405, 1254, 1253, 1741, 1924, 709, 460, 2189, 672, 174, 526, 1891, 3969]


In [20]:
POPULAR_ITEMS = np.array(
    [item for item, _ in pop.most_common()],
    dtype=np.int64,
)

POPULAR_WEIGHTS = np.array(
    [pop[int(item)] for item in POPULAR_ITEMS],
    dtype=np.float64,
)
POPULAR_WEIGHTS = POPULAR_WEIGHTS / POPULAR_WEIGHTS.sum()

POPULAR_ITEMS_CPU = torch.tensor(POPULAR_ITEMS, dtype=torch.long)
POPULAR_WEIGHTS_CPU = torch.tensor(POPULAR_WEIGHTS, dtype=torch.float)

if torch.cuda.is_available():
    POPULAR_ITEMS_GPU = POPULAR_ITEMS_CPU.to(DEVICE, non_blocking=True)
    POPULAR_WEIGHTS_GPU = POPULAR_WEIGHTS_CPU.to(DEVICE, non_blocking=True)
else:
    POPULAR_ITEMS_GPU = POPULAR_ITEMS_CPU
    POPULAR_WEIGHTS_GPU = POPULAR_WEIGHTS_CPU

## 9. Training datasets and SSL augmentations

This section defines:
- pairwise next-item training samples,
- CL4SRec-style sequence augmentations,
- SimDCL-style representation perturbation,
- and contrastive objectives used for SSL pretraining.

In [21]:
def sample_negative(n_items: int, forbidden: set[int], rng: random.Random) -> int:
    while True:
        j = rng.randint(1, n_items)
        if j not in forbidden:
            return j


class PairwiseNextItemDataset(Dataset):
    def __init__(
        self,
        train_seqs: Dict[int, List[int]],
        max_len: int,
        n_items: int,
        n_neg: int = 30,
        max_prefix_samples_per_user: Optional[int] = None,
        seed: int = 42,
    ):
        self.train_seqs = {
            u: np.asarray(seq, dtype=np.int64)
            for u, seq in train_seqs.items()
        }
        self.max_len = max_len
        self.n_items = n_items
        self.n_neg = n_neg
        self.seed = seed

        self.samples: List[Tuple[int, int]] = []
        rng = random.Random(seed)

        for u, seq in self.train_seqs.items():
            if len(seq) < 2:
                continue

            positions = list(range(1, len(seq)))
            if max_prefix_samples_per_user is not None and len(positions) > max_prefix_samples_per_user:
                positions = rng.sample(positions, max_prefix_samples_per_user)
                positions.sort()

            for t in positions:
                self.samples.append((u, t))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        u, t = self.samples[idx]
        seq = self.train_seqs[u]

        hist = seq[:t]
        pos = int(seq[t])

        hist_items = np.zeros(self.max_len, dtype=np.int64)
        hist_slice = hist[-self.max_len:]
        hist_items[:len(hist_slice)] = hist_slice

        row = {
            "user_id": np.int64(u),
            "seq_items": hist_items,
            "pos_item": np.int64(pos),
        }
        return row
    

def pairwise_collate_fn(batch):
    user_ids = torch.tensor([x["user_id"] for x in batch], dtype=torch.long)
    seq_items = torch.tensor(np.stack([x["seq_items"] for x in batch]), dtype=torch.long)
    pos_item = torch.tensor([x["pos_item"] for x in batch], dtype=torch.long)

    seq_artist_ids = ITEM_ARTIST_IDS_CPU[seq_items]
    seq_album_ids = ITEM_ALBUM_IDS_CPU[seq_items]
    seq_duration_ids = ITEM_DURATION_IDS_CPU[seq_items]
    seq_track_title_tokens = ITEM_TRACK_TITLE_TOKENS_CPU[seq_items]
    playlist_title_tokens = USER_PLAYLIST_TITLE_TOKENS_CPU[user_ids]

    pos_artist_id = ITEM_ARTIST_IDS_CPU[pos_item]
    pos_album_id = ITEM_ALBUM_IDS_CPU[pos_item]
    pos_duration_id = ITEM_DURATION_IDS_CPU[pos_item]
    pos_track_title_tokens = ITEM_TRACK_TITLE_TOKENS_CPU[pos_item]

    return {
        "user_id": user_ids,
        "seq_items": seq_items,
        "seq_artist_ids": seq_artist_ids,
        "seq_album_ids": seq_album_ids,
        "seq_duration_ids": seq_duration_ids,
        "seq_track_title_tokens": seq_track_title_tokens,
        "playlist_title_tokens": playlist_title_tokens,
        "pos_item": pos_item,
        "pos_artist_id": pos_artist_id,
        "pos_album_id": pos_album_id,
        "pos_duration_id": pos_duration_id,
        "pos_track_title_tokens": pos_track_title_tokens,
    }


@torch.no_grad()
def sample_negatives_on_device(
    seq_items: torch.Tensor,
    pos_item: torch.Tensor,
    n_items: int,
    n_neg: int,
    popular_frac: float = 0.5,
) -> torch.Tensor:
    device = seq_items.device
    batch_size = seq_items.size(0)

    n_pop = int(round(n_neg * popular_frac))
    n_rand = n_neg - n_pop

    parts = []

    if n_rand > 0:
        rand_negs = torch.randint(
            low=1,
            high=n_items + 1,
            size=(batch_size, n_rand),
            device=device,
        )
        parts.append(rand_negs)

    if n_pop > 0:
        pop_sample_idx = torch.multinomial(
            POPULAR_WEIGHTS_GPU,
            num_samples=batch_size * n_pop,
            replacement=True,
        ).view(batch_size, n_pop)

        pop_negs = POPULAR_ITEMS_GPU[pop_sample_idx]
        parts.append(pop_negs)

    neg_items = torch.cat(parts, dim=1)

    # Убираем positive и seen items
    seen_mask = (neg_items.unsqueeze(-1) == seq_items.unsqueeze(1)).any(dim=-1)
    pos_mask = neg_items.eq(pos_item.unsqueeze(1))
    bad_mask = seen_mask | pos_mask

    while bad_mask.any():
        n_bad = int(bad_mask.sum().item())

        refill_rand = torch.randint(
            low=1,
            high=n_items + 1,
            size=(n_bad,),
            device=device,
        )
        neg_items[bad_mask] = refill_rand

        seen_mask = (neg_items.unsqueeze(-1) == seq_items.unsqueeze(1)).any(dim=-1)
        pos_mask = neg_items.eq(pos_item.unsqueeze(1))
        bad_mask = seen_mask | pos_mask

    return neg_items

class ContrastiveDataset(Dataset):
    def __init__(self, train_seqs: Dict[int, List[int]], max_len: int):
        self.users = list(train_seqs.keys())
        self.train_seqs = {
            u: np.asarray(seq, dtype=np.int64)
            for u, seq in train_seqs.items()
        }
        self.max_len = max_len

    def __len__(self):
        return len(self.users)

    def _pack_view_np(self, seq: List[int]) -> np.ndarray:
        seq_arr = np.asarray(seq, dtype=np.int64)
        out = np.zeros(self.max_len, dtype=np.int64)
        seq_slice = seq_arr[-self.max_len:]
        out[:len(seq_slice)] = seq_slice
        return out

    def __getitem__(self, idx):
        u = self.users[idx]
        seq = self.train_seqs[u].tolist()

        v1 = make_view(seq)
        v2 = make_view(seq)

        row = {
            "user_id": np.int64(u),
            "view1_seq_items": self._pack_view_np(v1),
            "view2_seq_items": self._pack_view_np(v2),
        }
        return row
    

def ssl_collate_fn(batch):
    user_ids = torch.tensor([x["user_id"] for x in batch], dtype=torch.long)

    view1_seq_items = torch.tensor(
        np.stack([x["view1_seq_items"] for x in batch]),
        dtype=torch.long,
    )
    view2_seq_items = torch.tensor(
        np.stack([x["view2_seq_items"] for x in batch]),
        dtype=torch.long,
    )

    playlist_title_tokens = USER_PLAYLIST_TITLE_TOKENS_CPU[user_ids]

    v1 = {
        "seq_items": view1_seq_items,
        "seq_artist_ids": ITEM_ARTIST_IDS_CPU[view1_seq_items],
        "seq_album_ids": ITEM_ALBUM_IDS_CPU[view1_seq_items],
        "seq_duration_ids": ITEM_DURATION_IDS_CPU[view1_seq_items],
        "seq_track_title_tokens": ITEM_TRACK_TITLE_TOKENS_CPU[view1_seq_items],
        "playlist_title_tokens": playlist_title_tokens,
    }

    v2 = {
        "seq_items": view2_seq_items,
        "seq_artist_ids": ITEM_ARTIST_IDS_CPU[view2_seq_items],
        "seq_album_ids": ITEM_ALBUM_IDS_CPU[view2_seq_items],
        "seq_duration_ids": ITEM_DURATION_IDS_CPU[view2_seq_items],
        "seq_track_title_tokens": ITEM_TRACK_TITLE_TOKENS_CPU[view2_seq_items],
        "playlist_title_tokens": playlist_title_tokens,
    }

    return v1, v2


def make_ssl_loader(train_seqs):
    ds_ssl = ContrastiveDataset(train_seqs, MAX_SEQ_LEN)
    dl_ssl = DataLoader(
        ds_ssl,
        batch_size=BATCH_SIZE,
        shuffle=True,
        drop_last=False,
        num_workers=0,
        pin_memory=False,
        persistent_workers=False,
        collate_fn=ssl_collate_fn,
    )
    return ds_ssl, dl_ssl


def aug_crop(seq, min_ratio: float = 0.5):
    if len(seq) < 2:
        return seq.copy()
    L = len(seq)
    new_len = max(2, int(L * random.uniform(min_ratio, 1.0)))
    start = random.randint(0, L - new_len)
    return seq[start:start + new_len]


def aug_mask(seq, p: float = 0.2):
    if len(seq) <= 1:
        return seq.copy()

    out = []
    for x in seq:
        if random.random() >= p:
            out.append(x)

    if len(out) == 0:
        out = [random.choice(seq)]

    return out


def aug_reorder(seq, window: int = 3):
    out = seq.copy()
    if len(out) <= window:
        random.shuffle(out)
        return out
    start = random.randint(0, len(out) - window)
    sub = out[start:start + window]
    random.shuffle(sub)
    out[start:start + window] = sub
    return out


def make_view(seq):
    s = aug_crop(seq)
    s = aug_reorder(s, window=3)
    s = aug_mask(s, p=0.2)

    if len(s) == 0:
        s = [random.choice(seq)]

    return s


def info_nce_loss(z1: torch.Tensor, z2: torch.Tensor, temperature: float):
    z1 = F.normalize(z1, dim=-1)
    z2 = F.normalize(z2, dim=-1)

    logits_12 = (z1 @ z2.t()) / temperature
    logits_21 = (z2 @ z1.t()) / temperature
    labels = torch.arange(z1.size(0), device=z1.device)

    loss_12 = F.cross_entropy(logits_12, labels)
    loss_21 = F.cross_entropy(logits_21, labels)
    return 0.5 * (loss_12 + loss_21)


def simdcl_loss(z: torch.Tensor, noise_std: float, temperature: float):
    z1 = F.normalize(z, dim=-1)
    z2 = F.normalize(z + torch.randn_like(z) * noise_std, dim=-1)
    logits = (z1 @ z2.t()) / temperature
    labels = torch.arange(z1.size(0), device=z1.device)
    return F.cross_entropy(logits, labels)


def make_pairwise_loader(train_seqs, seed: int):
    ds = PairwiseNextItemDataset(
        train_seqs=train_seqs,
        max_len=MAX_SEQ_LEN,
        n_items=n_items,
        n_neg=30,
        max_prefix_samples_per_user=10,
        seed=seed,
    )
    dl = DataLoader(
        ds,
        batch_size=BATCH_SIZE,
        shuffle=True,
        drop_last=False,
        num_workers=0,
        pin_memory=False,
        persistent_workers=False,
        collate_fn=pairwise_collate_fn,
    )
    return ds, dl

## 10. Model architectures

This benchmark includes:
- TopPop
- GRU4Rec
- SASRec
- FMLP-Rec-style efficient baseline
- ComiRec-SA-style SASRec

Transformer-family models use both:
- causal attention masks
- padding masks

The multi-interest model uses:
- target-aware label routing during training
- diversity regularization between interest vectors

In [22]:
def build_causal_mask(seq_len: int, device: torch.device) -> torch.Tensor:
    return torch.triu(torch.ones(seq_len, seq_len, device=device), diagonal=1).bool()


def gather_last_nonpad(hidden: torch.Tensor, seq: torch.Tensor) -> torch.Tensor:
    idx = (seq != 0).sum(dim=1) - 1
    idx = idx.clamp_min(0)
    return hidden[torch.arange(hidden.size(0), device=hidden.device), idx]

In [ ]:
class MeanTextEncoder(nn.Module):
    def __init__(self, vocab_size: int, d_model: int, pad_idx: int = 0):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, d_model, padding_idx=pad_idx)
        self.ln = nn.LayerNorm(d_model)

    def forward(self, token_ids: torch.Tensor) -> torch.Tensor:
        x = self.emb(token_ids)
        mask = (token_ids != 0).float().unsqueeze(-1)
        denom = mask.sum(dim=-2).clamp_min(1.0)
        pooled = (x * mask).sum(dim=-2) / denom
        return self.ln(pooled)

In [24]:
class SequenceModelBase(nn.Module):
    def get_user_repr(
        self,
        seq_items: torch.Tensor,
        seq_artist_ids: torch.Tensor,
        seq_album_ids: torch.Tensor,
        seq_duration_ids: torch.Tensor,
        seq_track_title_tokens: torch.Tensor,
        playlist_title_tokens: torch.Tensor,
    ) -> torch.Tensor:
        raise NotImplementedError

    def score_items_from_features(
        self,
        user_repr: torch.Tensor,
        item_ids: torch.Tensor,
        artist_ids: torch.Tensor,
        album_ids: torch.Tensor,
        duration_ids: torch.Tensor,
        track_title_tokens: torch.Tensor,
    ) -> torch.Tensor:
        raise NotImplementedError

    def pooled_ssl_repr(
        self,
        seq_items: torch.Tensor,
        seq_artist_ids: torch.Tensor,
        seq_album_ids: torch.Tensor,
        seq_duration_ids: torch.Tensor,
        seq_track_title_tokens: torch.Tensor,
        playlist_title_tokens: torch.Tensor,
    ) -> torch.Tensor:
        return self.get_user_repr(
            seq_items,
            seq_artist_ids,
            seq_album_ids,
            seq_duration_ids,
            seq_track_title_tokens,
            playlist_title_tokens,
        )

In [25]:
class MultiInterestBase(nn.Module):
    def get_interest_repr(
        self,
        seq_items: torch.Tensor,
        seq_artist_ids: torch.Tensor,
        seq_album_ids: torch.Tensor,
        seq_duration_ids: torch.Tensor,
        seq_track_title_tokens: torch.Tensor,
        playlist_title_tokens: torch.Tensor,
    ) -> torch.Tensor:
        raise NotImplementedError

    def score_items_from_features(
        self,
        interest_repr: torch.Tensor,
        item_ids: torch.Tensor,
        artist_ids: torch.Tensor,
        album_ids: torch.Tensor,
        duration_ids: torch.Tensor,
        track_title_tokens: torch.Tensor,
    ) -> torch.Tensor:
        raise NotImplementedError

In [26]:
class GRU4Rec(SequenceModelBase):
    def __init__(self, n_items: int, d_model: int, hidden_size: int, n_layers: int, dropout: float):
        super().__init__()
        self.item_emb = nn.Embedding(n_items + 1, d_model, padding_idx=0)
        self.gru = nn.GRU(
            input_size=d_model,
            hidden_size=hidden_size,
            num_layers=n_layers,
            batch_first=True,
            dropout=dropout if n_layers > 1 else 0.0,
        )
        self.proj = nn.Linear(hidden_size, d_model)
        self.drop = nn.Dropout(dropout)
        self.ln = nn.LayerNorm(d_model)

    def get_user_repr(
        self,
        seq_items: torch.Tensor,
        seq_artist_ids: torch.Tensor,
        seq_album_ids: torch.Tensor,
        seq_duration_ids: torch.Tensor,
        seq_track_title_tokens: torch.Tensor,
        playlist_title_tokens: torch.Tensor,
    ) -> torch.Tensor:
        x = self.drop(self.item_emb(seq_items))
        h, _ = self.gru(x)
        out = gather_last_nonpad(h, seq_items)
        return self.ln(self.proj(out))

    def score_items_from_features(
        self,
        user_repr: torch.Tensor,
        item_ids: torch.Tensor,
        artist_ids: torch.Tensor,
        album_ids: torch.Tensor,
        duration_ids: torch.Tensor,
        track_title_tokens: torch.Tensor,
    ) -> torch.Tensor:
        v = self.item_emb(item_ids)
        return (v * user_repr.unsqueeze(1)).sum(-1)


class MetadataSASRec(SequenceModelBase):
    def __init__(
        self,
        n_items: int,
        n_artists: int,
        n_albums: int,
        n_durations: int,
        n_track_title_tokens: int,
        n_playlist_title_tokens: int,
        max_len: int,
        d_model: int,
        n_heads: int,
        n_layers: int,
        dropout: float,
    ):
        super().__init__()
        self.item_emb = nn.Embedding(n_items + 1, d_model, padding_idx=0)
        self.artist_emb = nn.Embedding(n_artists, d_model, padding_idx=0)
        self.album_emb = nn.Embedding(n_albums, d_model, padding_idx=0)
        self.duration_emb = nn.Embedding(n_durations, d_model, padding_idx=0)

        self.track_title_encoder = MeanTextEncoder(n_track_title_tokens, d_model, pad_idx=0)
        self.playlist_title_encoder = MeanTextEncoder(n_playlist_title_tokens, d_model, pad_idx=0)

        self.title_gate = nn.Sequential(
            nn.Linear(2 * d_model, d_model),
            nn.GELU(),
            nn.Linear(d_model, d_model),
            nn.Sigmoid(),
        )

        self.pos_emb = nn.Embedding(max_len, d_model)
        self.drop = nn.Dropout(dropout)

        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=n_heads,
            dim_feedforward=4 * d_model,
            dropout=dropout,
            batch_first=True,
            activation="gelu",
        )
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=n_layers)

        self.ln_item = nn.LayerNorm(d_model)
        self.ln_hidden = nn.LayerNorm(d_model)
        self.ln_out = nn.LayerNorm(d_model)

        self.ssl_proj = nn.Sequential(
            nn.Linear(d_model, d_model),
            nn.GELU(),
            nn.Linear(d_model, d_model),
        )

    def encode_item_features(
        self,
        item_ids: torch.Tensor,
        artist_ids: torch.Tensor,
        album_ids: torch.Tensor,
        duration_ids: torch.Tensor,
        track_title_tokens: torch.Tensor,
    ) -> torch.Tensor:
        item_vec = self.item_emb(item_ids)
        artist_vec = self.artist_emb(artist_ids)
        album_vec = self.album_emb(album_ids)
        duration_vec = self.duration_emb(duration_ids)

        orig_shape = track_title_tokens.shape[:-1]
        title_vec = self.track_title_encoder(
            track_title_tokens.reshape(-1, track_title_tokens.shape[-1])
        )
        title_vec = title_vec.reshape(*orig_shape, -1)

        meta_vec = (artist_vec + album_vec + duration_vec + title_vec) / 4.0
        fused = item_vec + 0.3 * meta_vec
        fused = torch.nan_to_num(fused, nan=0.0, posinf=0.0, neginf=0.0)
        return self.ln_item(fused)

    def forward_hidden(
        self,
        seq_items: torch.Tensor,
        seq_artist_ids: torch.Tensor,
        seq_album_ids: torch.Tensor,
        seq_duration_ids: torch.Tensor,
        seq_track_title_tokens: torch.Tensor,
    ) -> torch.Tensor:
        B, L = seq_items.shape
        pos = torch.arange(L, device=seq_items.device).unsqueeze(0).expand(B, L)

        x = self.encode_item_features(
            seq_items,
            seq_artist_ids,
            seq_album_ids,
            seq_duration_ids,
            seq_track_title_tokens,
        )
        x = x + self.pos_emb(pos)
        x = self.drop(x)

        pad_mask = seq_items == 0
        x = x.masked_fill(pad_mask.unsqueeze(-1), 0.0)

        attn_mask = build_causal_mask(L, seq_items.device)

        h = self.encoder(
            x,
            mask=attn_mask,
            src_key_padding_mask=pad_mask,
        )

        h = torch.nan_to_num(h, nan=0.0, posinf=0.0, neginf=0.0)
        h = h.masked_fill(pad_mask.unsqueeze(-1), 0.0)
        return self.ln_hidden(h)

    def get_seq_repr(
        self,
        seq_items: torch.Tensor,
        seq_artist_ids: torch.Tensor,
        seq_album_ids: torch.Tensor,
        seq_duration_ids: torch.Tensor,
        seq_track_title_tokens: torch.Tensor,
    ) -> torch.Tensor:
        h = self.forward_hidden(
            seq_items,
            seq_artist_ids,
            seq_album_ids,
            seq_duration_ids,
            seq_track_title_tokens,
        )
        seq_repr = gather_last_nonpad(h, seq_items)
        seq_repr = torch.nan_to_num(seq_repr, nan=0.0, posinf=0.0, neginf=0.0)
        return seq_repr

    def get_user_repr(
        self,
        seq_items: torch.Tensor,
        seq_artist_ids: torch.Tensor,
        seq_album_ids: torch.Tensor,
        seq_duration_ids: torch.Tensor,
        seq_track_title_tokens: torch.Tensor,
        playlist_title_tokens: torch.Tensor,
    ) -> torch.Tensor:
        seq_repr = self.get_seq_repr(
            seq_items,
            seq_artist_ids,
            seq_album_ids,
            seq_duration_ids,
            seq_track_title_tokens,
        )

        playlist_title_repr = self.playlist_title_encoder(playlist_title_tokens)
        playlist_title_repr = torch.nan_to_num(playlist_title_repr, nan=0.0, posinf=0.0, neginf=0.0)

        gate = self.title_gate(torch.cat([seq_repr, playlist_title_repr], dim=-1))
        out = seq_repr + gate * playlist_title_repr
        out = torch.nan_to_num(out, nan=0.0, posinf=0.0, neginf=0.0)
        return self.ln_out(out)

    def pooled_ssl_repr(
        self,
        seq_items: torch.Tensor,
        seq_artist_ids: torch.Tensor,
        seq_album_ids: torch.Tensor,
        seq_duration_ids: torch.Tensor,
        seq_track_title_tokens: torch.Tensor,
        playlist_title_tokens: torch.Tensor,
    ) -> torch.Tensor:
        seq_repr = self.get_seq_repr(
            seq_items,
            seq_artist_ids,
            seq_album_ids,
            seq_duration_ids,
            seq_track_title_tokens,
        )
        z = self.ssl_proj(seq_repr)
        z = torch.nan_to_num(z, nan=0.0, posinf=0.0, neginf=0.0)
        return z

    def score_items_from_features(
        self,
        user_repr: torch.Tensor,
        item_ids: torch.Tensor,
        artist_ids: torch.Tensor,
        album_ids: torch.Tensor,
        duration_ids: torch.Tensor,
        track_title_tokens: torch.Tensor,
    ) -> torch.Tensor:
        item_repr = self.encode_item_features(
            item_ids,
            artist_ids,
            album_ids,
            duration_ids,
            track_title_tokens,
        )
        scores = (item_repr * user_repr.unsqueeze(1)).sum(dim=-1)
        return torch.nan_to_num(scores, nan=-1e9, posinf=1e9, neginf=-1e9)


class FMLPBlock(nn.Module):
    def __init__(self, d_model: int, dropout: float):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.filter = nn.Sequential(
            nn.Linear(d_model, d_model),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model, d_model),
        )
        self.ln2 = nn.LayerNorm(d_model)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, 4 * d_model),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(4 * d_model, d_model),
            nn.Dropout(dropout),
        )

    def forward(self, x: torch.Tensor, pad_mask: torch.Tensor) -> torch.Tensor:
        y = self.filter(self.ln1(x))
        x = x + y
        x = x.masked_fill(pad_mask.unsqueeze(-1), 0.0)
        z = self.ffn(self.ln2(x))
        x = x + z
        x = x.masked_fill(pad_mask.unsqueeze(-1), 0.0)
        return x


class FMLPRec(SequenceModelBase):
    def __init__(self, n_items: int, max_len: int, d_model: int, n_layers: int, dropout: float):
        super().__init__()
        self.item_emb = nn.Embedding(n_items + 1, d_model, padding_idx=0)
        self.pos_emb = nn.Embedding(max_len, d_model)
        self.drop = nn.Dropout(dropout)
        self.blocks = nn.ModuleList([FMLPBlock(d_model=d_model, dropout=dropout) for _ in range(n_layers)])
        self.ln = nn.LayerNorm(d_model)

    def forward_hidden(self, seq_items: torch.Tensor) -> torch.Tensor:
        B, L = seq_items.shape
        pos = torch.arange(L, device=seq_items.device).unsqueeze(0).expand(B, L)
        x = self.item_emb(seq_items) + self.pos_emb(pos)
        x = self.drop(x)
        pad_mask = (seq_items == 0)

        x = x.masked_fill(pad_mask.unsqueeze(-1), 0.0)
        for block in self.blocks:
            x = block(x, pad_mask)
        return self.ln(x)

    def get_user_repr(
        self,
        seq_items: torch.Tensor,
        seq_artist_ids: torch.Tensor,
        seq_album_ids: torch.Tensor,
        seq_duration_ids: torch.Tensor,
        seq_track_title_tokens: torch.Tensor,
        playlist_title_tokens: torch.Tensor,
    ) -> torch.Tensor:
        h = self.forward_hidden(seq_items)
        return gather_last_nonpad(h, seq_items)

    def score_items_from_features(
        self,
        user_repr: torch.Tensor,
        item_ids: torch.Tensor,
        artist_ids: torch.Tensor,
        album_ids: torch.Tensor,
        duration_ids: torch.Tensor,
        track_title_tokens: torch.Tensor,
    ) -> torch.Tensor:
        v = self.item_emb(item_ids)
        return (v * user_repr.unsqueeze(1)).sum(-1)


class MetadataComiRecSA(MultiInterestBase):
    def __init__(
        self,
        n_items: int,
        n_artists: int,
        n_albums: int,
        n_durations: int,
        n_track_title_tokens: int,
        n_playlist_title_tokens: int,
        max_len: int,
        d_model: int,
        n_heads: int,
        n_layers: int,
        dropout: float,
        n_interests: int,
    ):
        super().__init__()
        self.n_interests = n_interests

        self.item_emb = nn.Embedding(n_items + 1, d_model, padding_idx=0)
        self.artist_emb = nn.Embedding(n_artists, d_model, padding_idx=0)
        self.album_emb = nn.Embedding(n_albums, d_model, padding_idx=0)
        self.duration_emb = nn.Embedding(n_durations, d_model, padding_idx=0)

        self.track_title_encoder = MeanTextEncoder(n_track_title_tokens, d_model, pad_idx=0)
        self.playlist_title_encoder = MeanTextEncoder(n_playlist_title_tokens, d_model, pad_idx=0)

        self.pos_emb = nn.Embedding(max_len, d_model)
        self.drop = nn.Dropout(dropout)

        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=n_heads,
            dim_feedforward=4 * d_model,
            dropout=dropout,
            batch_first=True,
            activation="gelu",
        )
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=n_layers)

        self.ln_item = nn.LayerNorm(d_model)
        self.ln_hidden = nn.LayerNorm(d_model)

        self.interest_queries = nn.Parameter(torch.randn(n_interests, d_model) * 0.02)
        self.playlist_title_proj = nn.Linear(d_model, d_model)

    def encode_item_features(
        self,
        item_ids: torch.Tensor,
        artist_ids: torch.Tensor,
        album_ids: torch.Tensor,
        duration_ids: torch.Tensor,
        track_title_tokens: torch.Tensor,
    ) -> torch.Tensor:
        item_vec = self.item_emb(item_ids)
        artist_vec = self.artist_emb(artist_ids)
        album_vec = self.album_emb(album_ids)
        duration_vec = self.duration_emb(duration_ids)

        orig_shape = track_title_tokens.shape[:-1]
        title_vec = self.track_title_encoder(
            track_title_tokens.reshape(-1, track_title_tokens.shape[-1])
        )
        title_vec = title_vec.reshape(*orig_shape, -1)

        meta_vec = (artist_vec + album_vec + duration_vec + title_vec) / 4.0
        fused = item_vec + 0.3 * meta_vec
        fused = torch.nan_to_num(fused, nan=0.0, posinf=0.0, neginf=0.0)
        return self.ln_item(fused)

    def forward_hidden(
        self,
        seq_items: torch.Tensor,
        seq_artist_ids: torch.Tensor,
        seq_album_ids: torch.Tensor,
        seq_duration_ids: torch.Tensor,
        seq_track_title_tokens: torch.Tensor,
    ) -> torch.Tensor:
        B, L = seq_items.shape
        pos = torch.arange(L, device=seq_items.device).unsqueeze(0).expand(B, L)

        x = self.encode_item_features(
            seq_items,
            seq_artist_ids,
            seq_album_ids,
            seq_duration_ids,
            seq_track_title_tokens,
        )
        x = x + self.pos_emb(pos)
        x = self.drop(x)

        pad_mask = seq_items == 0
        x = x.masked_fill(pad_mask.unsqueeze(-1), 0.0)

        attn_mask = build_causal_mask(L, seq_items.device)

        h = self.encoder(
            x,
            mask=attn_mask,
            src_key_padding_mask=pad_mask,
        )

        h = torch.nan_to_num(h, nan=0.0, posinf=0.0, neginf=0.0)
        h = h.masked_fill(pad_mask.unsqueeze(-1), 0.0)
        return self.ln_hidden(h)

    def get_interest_repr(
        self,
        seq_items: torch.Tensor,
        seq_artist_ids: torch.Tensor,
        seq_album_ids: torch.Tensor,
        seq_duration_ids: torch.Tensor,
        seq_track_title_tokens: torch.Tensor,
        playlist_title_tokens: torch.Tensor,
    ) -> torch.Tensor:
        h = self.forward_hidden(
            seq_items,
            seq_artist_ids,
            seq_album_ids,
            seq_duration_ids,
            seq_track_title_tokens,
        )
        mask = seq_items != 0

        B, L, D = h.shape
        q = self.interest_queries.unsqueeze(0).expand(B, self.n_interests, D)

        attn_logits = torch.matmul(q, h.transpose(1, 2)) / math.sqrt(D)
        attn_logits = attn_logits.masked_fill(~mask.unsqueeze(1), -1e9)

        attn = torch.softmax(attn_logits, dim=-1)
        attn = torch.nan_to_num(attn, nan=0.0, posinf=0.0, neginf=0.0)

        z = torch.matmul(attn, h)

        playlist_title_repr = self.playlist_title_encoder(playlist_title_tokens)
        playlist_title_repr = torch.nan_to_num(playlist_title_repr, nan=0.0, posinf=0.0, neginf=0.0)

        z = z + self.playlist_title_proj(playlist_title_repr).unsqueeze(1)
        z = torch.nan_to_num(z, nan=0.0, posinf=0.0, neginf=0.0)

        z = F.normalize(z, dim=-1, eps=1e-8)
        z = torch.nan_to_num(z, nan=0.0, posinf=0.0, neginf=0.0)
        return z

    def score_items_from_features(
        self,
        interest_repr: torch.Tensor,
        item_ids: torch.Tensor,
        artist_ids: torch.Tensor,
        album_ids: torch.Tensor,
        duration_ids: torch.Tensor,
        track_title_tokens: torch.Tensor,
    ) -> torch.Tensor:
        item_repr = self.encode_item_features(
            item_ids,
            artist_ids,
            album_ids,
            duration_ids,
            track_title_tokens,
        )
        scores = torch.einsum("bkd,bcd->bkc", interest_repr, item_repr)
        scores = scores.max(dim=1).values
        return torch.nan_to_num(scores, nan=-1e9, posinf=1e9, neginf=-1e9)

    def score_items_per_interest_from_features(
        self,
        interest_repr: torch.Tensor,
        item_ids: torch.Tensor,
        artist_ids: torch.Tensor,
        album_ids: torch.Tensor,
        duration_ids: torch.Tensor,
        track_title_tokens: torch.Tensor,
    ) -> torch.Tensor:
        item_repr = self.encode_item_features(
            item_ids,
            artist_ids,
            album_ids,
            duration_ids,
            track_title_tokens,
        )
        scores = torch.einsum("bkd,bcd->bkc", interest_repr, item_repr)
        return torch.nan_to_num(scores, nan=0.0, posinf=0.0, neginf=0.0)

In [27]:
def predict_toppop_ranked(
    eval_cases: EvalCases,
    train_or_eval_seqs: Dict[int, List[int]],
    pop_counter: Counter,
    n_items: int,
    k: int = 20,
) -> np.ndarray:
    item_ids = np.arange(1, n_items + 1, dtype=np.int32)

    global_scores = np.asarray(
        [pop_counter.get(int(i), 0) for i in item_ids],
        dtype=np.int32,
    )

    global_order = np.lexsort((item_ids, -global_scores))
    global_ranked_items = item_ids[global_order]

    ranked = np.zeros((len(eval_cases.users), k), dtype=np.int32)

    for row_idx, u in enumerate(eval_cases.users):
        seen_items = set(train_or_eval_seqs[u][:-1])

        out = []
        for item in global_ranked_items:
            if item not in seen_items:
                out.append(item)
                if len(out) == k:
                    break

        ranked[row_idx, :] = out

    return ranked


def evaluate_toppop_full_ranking(
    eval_cases: EvalCases,
    train_or_eval_seqs: Dict[int, List[int]],
    pop_counter: Counter,
    n_items: int,
    k: int = 20,
):
    ranked = predict_toppop_ranked(
        eval_cases=eval_cases,
        train_or_eval_seqs=train_or_eval_seqs,
        pop_counter=pop_counter,
        n_items=n_items,
        k=k,
    )
    metrics = compute_metrics_from_ranked(ranked, eval_cases.targets, n_items, k)
    return metrics[f"Recall@{k}"], metrics[f"NDCG@{k}"], metrics[f"MRR@{k}"], metrics[f"Coverage@{k}"]

In [ ]:
def build_eval_batch_tensors(
    batch_users: List[int],
    batch_histories: np.ndarray,
) -> Dict[str, torch.Tensor]:
    seq_items = torch.as_tensor(
        batch_histories,
        dtype=torch.long,
        device=DEVICE,
    )
    batch_users_t = torch.as_tensor(
        batch_users,
        dtype=torch.long,
        device=DEVICE,
    )

    return {
        "seq_items": seq_items,
        "seq_artist_ids": ITEM_ARTIST_IDS_GPU[seq_items],
        "seq_album_ids": ITEM_ALBUM_IDS_GPU[seq_items],
        "seq_duration_ids": ITEM_DURATION_IDS_GPU[seq_items],
        "seq_track_title_tokens": ITEM_TRACK_TITLE_TOKENS_GPU[seq_items],
        "playlist_title_tokens": USER_PLAYLIST_TITLE_TOKENS_GPU[batch_users_t],
    }


def build_all_item_feature_tensors_cpu() -> Dict[str, torch.Tensor]:
    return {
        "item_ids": torch.arange(1, n_items + 1, dtype=torch.long),
        "artist_ids": torch.tensor([itemid_to_artist_id.get(i, 0) for i in range(1, n_items + 1)], dtype=torch.long),
        "album_ids": torch.tensor([itemid_to_album_id.get(i, 0) for i in range(1, n_items + 1)], dtype=torch.long),
        "duration_ids": torch.tensor([itemid_to_duration_id.get(i, 0) for i in range(1, n_items + 1)], dtype=torch.long),
        "track_title_tokens": torch.tensor(
            [itemid_to_track_title_tokens.get(i, [0] * ITEM_TITLE_MAX_TOKENS) for i in range(1, n_items + 1)],
            dtype=torch.long,
        ),
    }


ALL_ITEM_FEATURES_CPU = build_all_item_feature_tensors_cpu()


def get_item_matrix_cache_key(
    model: nn.Module,
    cache_key: Optional[str],
    item_batch_size: int,
) -> str:
    if cache_key is not None:
        return f"{cache_key}::items::{item_batch_size}"
    return f"{type(model).__name__}::{id(model)}::items::{item_batch_size}"


@torch.no_grad()
def encode_all_items_for_model(
    model: nn.Module,
    item_batch_size: int = 8192,
    cache_key: Optional[str] = None,
) -> torch.Tensor:
    model.eval()

    effective_cache_key = get_item_matrix_cache_key(
        model=model,
        cache_key=cache_key,
        item_batch_size=item_batch_size,
    )

    if effective_cache_key in ITEM_MATRIX_CACHE:
        return ITEM_MATRIX_CACHE[effective_cache_key]

    item_ids_cpu = ALL_ITEM_FEATURES_CPU["item_ids"]
    artist_ids_cpu = ALL_ITEM_FEATURES_CPU["artist_ids"]
    album_ids_cpu = ALL_ITEM_FEATURES_CPU["album_ids"]
    duration_ids_cpu = ALL_ITEM_FEATURES_CPU["duration_ids"]
    track_title_tokens_cpu = ALL_ITEM_FEATURES_CPU["track_title_tokens"]

    chunks = []
    for start in range(0, n_items, item_batch_size):
        end = min(start + item_batch_size, n_items)

        item_ids = item_ids_cpu[start:end].to(DEVICE, non_blocking=True)
        artist_ids = artist_ids_cpu[start:end].to(DEVICE, non_blocking=True)
        album_ids = album_ids_cpu[start:end].to(DEVICE, non_blocking=True)
        duration_ids = duration_ids_cpu[start:end].to(DEVICE, non_blocking=True)
        track_title_tokens = track_title_tokens_cpu[start:end].to(DEVICE, non_blocking=True)

        if isinstance(model, (MetadataSASRec, MetadataComiRecSA)):
            item_repr = model.encode_item_features(
                item_ids,
                artist_ids,
                album_ids,
                duration_ids,
                track_title_tokens,
            )
        elif isinstance(model, (GRU4Rec, FMLPRec)):
            item_repr = model.item_emb(item_ids)
        else:
            raise TypeError(f"Unsupported model type for item encoding: {type(model).__name__}")

        item_repr = torch.nan_to_num(item_repr, nan=0.0, posinf=0.0, neginf=0.0)
        chunks.append(item_repr)

    item_matrix = torch.cat(chunks, dim=0)
    ITEM_MATRIX_CACHE[effective_cache_key] = item_matrix
    return item_matrix


def ranked_array_to_dict(ranked: np.ndarray, users: List[int]) -> Dict[int, List[int]]:
    out: Dict[int, List[int]] = {}
    for u, row in zip(users, ranked):
        out[u] = [int(x) for x in row.tolist() if int(x) > 0]
    return out


def ranked_dict_to_array(
    ranked_dict: Dict[int, List[int]],
    users: List[int],
    k: int,
) -> np.ndarray:
    arr = np.zeros((len(users), k), dtype=np.int64)
    for i, u in enumerate(users):
        vals = ranked_dict.get(u, [])[:k]
        if vals:
            arr[i, :len(vals)] = np.asarray(vals, dtype=np.int64)
    return arr


def compute_metrics_from_ranked_dict(
    ranked_dict: Dict[int, List[int]],
    eval_cases: EvalCases,
    n_items: int,
    k: int,
) -> Dict[str, float]:
    arr = ranked_dict_to_array(ranked_dict, eval_cases.users, k)
    return compute_metrics_from_ranked(arr, eval_cases.targets, n_items, k)


@torch.no_grad()
def predict_repr_model_ranked(
    model: SequenceModelBase,
    eval_cases: EvalCases,
    eval_seqs: Dict[int, List[int]],
    n_items: int,
    k: int = 20,
    eval_batch_size: int = 256,
    item_chunk_size: int = 8192,
    cache_key: Optional[str] = None,
) -> np.ndarray:
    model.eval()
    ranked_all = []

    item_matrix = encode_all_items_for_model(
        model,
        item_batch_size=item_chunk_size,
        cache_key=cache_key,
    )

    for start in range(0, len(eval_cases.users), eval_batch_size):
        end = min(start + eval_batch_size, len(eval_cases.users))
        batch_users = eval_cases.users[start:end]
        batch_histories = eval_cases.histories[start:end]

        batch = build_eval_batch_tensors(batch_users, batch_histories)

        user_repr = model.get_user_repr(
            batch["seq_items"],
            batch["seq_artist_ids"],
            batch["seq_album_ids"],
            batch["seq_duration_ids"],
            batch["seq_track_title_tokens"],
            batch["playlist_title_tokens"],
        )

        scores = torch.matmul(user_repr, item_matrix.T)

        for i, u in enumerate(batch_users):
            seen_items = set(eval_seqs[u][:-1])
            if seen_items:
                seen_idx = [x - 1 for x in seen_items if 1 <= x <= n_items]
                scores[i, seen_idx] = -1e9

        _, top_idx = torch.topk(scores, k=k, dim=1)
        top_items = top_idx + 1
        ranked_all.append(top_items.cpu().numpy())

    return np.vstack(ranked_all)


@torch.no_grad()
def evaluate_repr_model_full_ranking(
    model: SequenceModelBase,
    eval_cases: EvalCases,
    eval_seqs: Dict[int, List[int]],
    n_items: int,
    k: int = 20,
    eval_batch_size: int = 256,
    item_chunk_size: int = 8192,
):
    ranked = predict_repr_model_ranked(
        model=model,
        eval_cases=eval_cases,
        eval_seqs=eval_seqs,
        n_items=n_items,
        k=k,
        eval_batch_size=eval_batch_size,
        item_chunk_size=item_chunk_size,
    )
    metrics = compute_metrics_from_ranked(ranked, eval_cases.targets, n_items, k)
    return (
        metrics[f"Recall@{k}"],
        metrics[f"NDCG@{k}"],
        metrics[f"MRR@{k}"],
        metrics[f"Coverage@{k}"],
    )


@torch.no_grad()
def predict_multi_interest_ranked(
    model: MultiInterestBase,
    eval_cases: EvalCases,
    eval_seqs: Dict[int, List[int]],
    n_items: int,
    k: int = 20,
    eval_batch_size: int = 256,
    item_chunk_size: int = 8192,
    cache_key: Optional[str] = None,
) -> np.ndarray:
    model.eval()
    ranked_all = []

    item_matrix = encode_all_items_for_model(
        model,
        item_batch_size=item_chunk_size,
        cache_key=cache_key,
    )

    for start in range(0, len(eval_cases.users), eval_batch_size):
        end = min(start + eval_batch_size, len(eval_cases.users))
        batch_users = eval_cases.users[start:end]
        batch_histories = eval_cases.histories[start:end]

        batch = build_eval_batch_tensors(batch_users, batch_histories)

        z = model.get_interest_repr(
            batch["seq_items"],
            batch["seq_artist_ids"],
            batch["seq_album_ids"],
            batch["seq_duration_ids"],
            batch["seq_track_title_tokens"],
            batch["playlist_title_tokens"],
        )

        scores = torch.einsum("bkd,nd->bkn", z, item_matrix).max(dim=1).values

        for i, u in enumerate(batch_users):
            seen_items = set(eval_seqs[u][:-1])
            if seen_items:
                seen_idx = [x - 1 for x in seen_items if 1 <= x <= n_items]
                scores[i, seen_idx] = -1e9

        _, top_idx = torch.topk(scores, k=k, dim=1)
        top_items = top_idx + 1
        ranked_all.append(top_items.cpu().numpy())

    return np.vstack(ranked_all)


@torch.no_grad()
def evaluate_multi_interest_full_ranking(
    model: MultiInterestBase,
    eval_cases: EvalCases,
    eval_seqs: Dict[int, List[int]],
    n_items: int,
    k: int = 20,
    eval_batch_size: int = 256,
    item_chunk_size: int = 8192,
):
    ranked = predict_multi_interest_ranked(
        model=model,
        eval_cases=eval_cases,
        eval_seqs=eval_seqs,
        n_items=n_items,
        k=k,
        eval_batch_size=eval_batch_size,
        item_chunk_size=item_chunk_size,
    )
    metrics = compute_metrics_from_ranked(ranked, eval_cases.targets, n_items, k)
    return (
        metrics[f"Recall@{k}"],
        metrics[f"NDCG@{k}"],
        metrics[f"MRR@{k}"],
        metrics[f"Coverage@{k}"],
    )


def build_sparse_inverted_index(
    item_tf_vecs: Dict[int, Dict[str, int]]
) -> Dict[str, List[Tuple[int, int]]]:
    inv: Dict[str, List[Tuple[int, int]]] = {}
    for item_id, vec in item_tf_vecs.items():
        for tok, tf in vec.items():
            inv.setdefault(tok, []).append((item_id, tf))
    return inv


SPARSE_INVERTED_INDEX = build_sparse_inverted_index(itemid_to_tf_vec)


def build_sparse_query_from_user_context(
    user_id: int,
    history_items: List[int],
    title_weight: float = 2.0,
    history_weight: float = 1.0,
    history_last_n: int = 20,
) -> Dict[str, float]:
    query: Counter = Counter()

    title_text = userid_to_playlist_title_text.get(user_id, "")
    title_tokens = tokenize_text_simple(title_text)
    for tok in title_tokens:
        query[tok] += title_weight

    history_slice = history_items[-history_last_n:]
    for item_id in history_slice:
        text = itemid_to_lexical_text.get(item_id, "")
        toks = tokenize_text_simple(text)
        for tok in toks:
            query[tok] += history_weight

    return {tok: float(weight) for tok, weight in query.items()}


def build_sparse_candidate_dict(
    eval_cases: EvalCases,
    eval_seqs: Dict[int, List[int]],
    top_m: int,
    title_weight: float,
    history_weight: float,
    history_last_n: int,
) -> Dict[int, List[int]]:
    out: Dict[int, List[int]] = {}

    for u in eval_cases.users:
        history_items = eval_seqs[u][:-1]
        seen_items = set(history_items)

        query_vec = build_sparse_query_from_user_context(
            user_id=u,
            history_items=history_items,
            title_weight=title_weight,
            history_weight=history_weight,
            history_last_n=history_last_n,
        )

        score_accum: Dict[int, float] = {}

        for tok, q_weight in query_vec.items():
            postings = SPARSE_INVERTED_INDEX.get(tok, [])
            df_val = LEXICAL_DF.get(tok, 0)
            if df_val <= 0:
                continue

            idf = bm25_idf(df_val=df_val, n_docs=LEXICAL_N_DOCS)

            for item_id, tf in postings:
                if item_id in seen_items:
                    continue

                dl = itemid_to_doc_len.get(item_id, 0)
                denom = tf + BM25_K1 * (1.0 - BM25_B + BM25_B * dl / max(BM25_AVGDL, 1e-9))
                bm25_term = idf * (tf * (BM25_K1 + 1.0)) / max(denom, 1e-9)

                score_accum[item_id] = score_accum.get(item_id, 0.0) + q_weight * bm25_term

        ranked = sorted(score_accum.items(), key=lambda x: (-x[1], x[0]))
        out[u] = [item_id for item_id, _ in ranked[:top_m]]

    return out


def build_dense_candidate_dict(
    model: nn.Module,
    eval_cases: EvalCases,
    eval_seqs: Dict[int, List[int]],
    top_m: int,
    eval_batch_size: int,
    item_chunk_size: int,
    cache_key: Optional[str] = None,
) -> Dict[int, List[int]]:
    if isinstance(model, MultiInterestBase):
        ranked = predict_multi_interest_ranked(
            model=model,
            eval_cases=eval_cases,
            eval_seqs=eval_seqs,
            n_items=n_items,
            k=top_m,
            eval_batch_size=eval_batch_size,
            item_chunk_size=item_chunk_size,
            cache_key=cache_key,
        )
    else:
        ranked = predict_repr_model_ranked(
            model=model,
            eval_cases=eval_cases,
            eval_seqs=eval_seqs,
            n_items=n_items,
            k=top_m,
            eval_batch_size=eval_batch_size,
            item_chunk_size=item_chunk_size,
            cache_key=cache_key,
        )
    return ranked_array_to_dict(ranked, eval_cases.users)


def hybrid_fuse_candidates(
    dense_candidates: List[int],
    sparse_candidates: List[int],
    dense_weight: float = 1.0,
    sparse_weight: float = 1.0,
    top_m: int = 200,
) -> List[int]:
    score: Dict[int, float] = {}

    for rank, item in enumerate(dense_candidates):
        score[item] = score.get(item, 0.0) + dense_weight / (rank + 1)

    for rank, item in enumerate(sparse_candidates):
        score[item] = score.get(item, 0.0) + sparse_weight / (rank + 1)

    fused = sorted(score.items(), key=lambda x: (-x[1], x[0]))
    return [item for item, _ in fused[:top_m]]


def build_hybrid_candidate_dict(
    dense_dict: Dict[int, List[int]],
    sparse_dict: Dict[int, List[int]],
    dense_weight: float,
    sparse_weight: float,
    top_m: int,
) -> Dict[int, List[int]]:
    out: Dict[int, List[int]] = {}
    users = sorted(set(dense_dict.keys()) | set(sparse_dict.keys()))

    for u in users:
        out[u] = hybrid_fuse_candidates(
            dense_dict.get(u, []),
            sparse_dict.get(u, []),
            dense_weight=dense_weight,
            sparse_weight=sparse_weight,
            top_m=top_m,
        )

    return out


def candidate_recall_at_m(
    candidate_dict: Dict[int, List[int]],
    eval_cases: EvalCases,
    m: int,
) -> float:
    hits = 0
    total = 0

    for u, target in zip(eval_cases.users, eval_cases.targets):
        total += 1
        candidates = candidate_dict.get(u, [])[:m]
        hits += int(int(target) in candidates)

    return float(hits / max(1, total))

## 11. Full-ranking, retrieval, and shortlist-reranking utilities

This section contains the shared utilities for:
- chunked full-ranking evaluation
- dense candidate generation
- sparse lexical candidate generation
- hybrid sparse+dense candidate fusion
- shortlist reranking
- IRRT metrics such as candidate recall and reranked ranking quality

In [29]:
@torch.no_grad()
def analyze_interest_usage(model: MultiInterestBase, eval_cases: EvalCases, eval_batch_size: int = 256):
    model.eval()
    all_best_interest, all_interest_entropy = [], []

    for start in range(0, len(eval_cases.users), eval_batch_size):
        end = min(start + eval_batch_size, len(eval_cases.users))
        batch_users = eval_cases.users[start:end]
        batch_histories = eval_cases.histories[start:end]

        batch = build_eval_batch_tensors(batch_users, batch_histories)

        targets = eval_cases.targets[start:end]
        target_item = torch.tensor(targets, dtype=torch.long, device=DEVICE)
        target_artist = torch.tensor([itemid_to_artist_id.get(x, 0) for x in targets], dtype=torch.long, device=DEVICE)
        target_album = torch.tensor([itemid_to_album_id.get(x, 0) for x in targets], dtype=torch.long, device=DEVICE)
        target_duration = torch.tensor([itemid_to_duration_id.get(x, 0) for x in targets], dtype=torch.long, device=DEVICE)
        target_title = torch.tensor(
            [itemid_to_track_title_tokens.get(x, [0] * ITEM_TITLE_MAX_TOKENS) for x in targets],
            dtype=torch.long,
            device=DEVICE,
        )

        z = model.get_interest_repr(
            batch["seq_items"],
            batch["seq_artist_ids"],
            batch["seq_album_ids"],
            batch["seq_duration_ids"],
            batch["seq_track_title_tokens"],
            batch["playlist_title_tokens"],
        )

        interest_scores = model.score_items_per_interest_from_features(
            z,
            target_item.unsqueeze(1),
            target_artist.unsqueeze(1),
            target_album.unsqueeze(1),
            target_duration.unsqueeze(1),
            target_title.unsqueeze(1),
        ).squeeze(-1)

        best_interest = interest_scores.argmax(dim=1).cpu().numpy()
        probs = torch.softmax(interest_scores, dim=1)
        entropy = (-(probs * probs.clamp_min(1e-9).log()).sum(dim=1)).cpu().numpy()

        all_best_interest.extend(best_interest.tolist())
        all_interest_entropy.extend(entropy.tolist())

    counts = np.bincount(np.asarray(all_best_interest), minlength=model.n_interests).astype(np.float64)
    counts = counts / counts.sum()

    return {
        "interest_usage_entropy": float(-(counts * np.log(counts + 1e-12)).sum()),
        "interest_usage_max_share": float(counts.max()),
        "interest_target_score_entropy_mean": float(np.mean(all_interest_entropy)),
    }


class EarlyStopper:
    def __init__(self, patience: int = 5, min_delta: float = 1e-4, mode: str = "max"):
        assert mode in ("max", "min")
        self.patience = patience
        self.min_delta = min_delta
        self.mode = mode
        self.best = None
        self.bad_epochs = 0
        self.best_state = None

    def _is_improvement(self, value: float) -> bool:
        if self.best is None:
            return True
        if self.mode == "max":
            return value > self.best + self.min_delta
        return value < self.best - self.min_delta

    def step(self, value: float, model: nn.Module) -> bool:
        if self._is_improvement(value):
            self.best = value
            self.bad_epochs = 0
            self.best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            return False
        self.bad_epochs += 1
        return self.bad_epochs >= self.patience

    def restore_best(self, model: nn.Module) -> None:
        if self.best_state is not None:
            model.load_state_dict(self.best_state)


def bpr_loss(pos_scores: torch.Tensor, neg_scores: torch.Tensor) -> torch.Tensor:
    return -torch.log(torch.sigmoid(pos_scores.unsqueeze(1) - neg_scores) + 1e-9).mean()


def diversity_regularization(z: torch.Tensor) -> torch.Tensor:
    z = F.normalize(z, dim=-1)
    sim = torch.matmul(z, z.transpose(1, 2))
    K = sim.size(1)
    eye = torch.eye(K, device=sim.device).unsqueeze(0)
    off_diag = sim * (1.0 - eye)
    return (off_diag ** 2).mean()


def label_aware_comirec_loss(
    model: MetadataComiRecSA,
    batch: Dict[str, torch.Tensor],
    label_tau: float,
    diversity_reg_weight: float,
    use_label_aware: bool = True,
) -> Tuple[torch.Tensor, Dict[str, float]]:
    z = model.get_interest_repr(
        batch["seq_items"],
        batch["seq_artist_ids"],
        batch["seq_album_ids"],
        batch["seq_duration_ids"],
        batch["seq_track_title_tokens"],
        batch["playlist_title_tokens"],
    )

    pos_scores_per_interest = model.score_items_per_interest_from_features(
        z,
        batch["pos_item"].unsqueeze(1),
        batch["pos_artist_id"].unsqueeze(1),
        batch["pos_album_id"].unsqueeze(1),
        batch["pos_duration_id"].unsqueeze(1),
        batch["pos_track_title_tokens"].unsqueeze(1),
    ).squeeze(-1)

    neg_scores_per_interest = model.score_items_per_interest_from_features(
        z,
        batch["neg_items"],
        batch["neg_artist_ids"],
        batch["neg_album_ids"],
        batch["neg_duration_ids"],
        batch["neg_track_title_tokens"],
    )

    pos_scores_per_interest = torch.nan_to_num(pos_scores_per_interest, nan=0.0, posinf=0.0, neginf=0.0)
    neg_scores_per_interest = torch.nan_to_num(neg_scores_per_interest, nan=0.0, posinf=0.0, neginf=0.0)

    if use_label_aware:
        routing_weights = torch.softmax(pos_scores_per_interest / max(label_tau, 1e-6), dim=1)
        routing_weights = torch.nan_to_num(routing_weights, nan=0.0, posinf=0.0, neginf=0.0)
        pos_scores = (routing_weights * pos_scores_per_interest).sum(dim=1)
        neg_scores = (routing_weights.unsqueeze(-1) * neg_scores_per_interest).sum(dim=1)
    else:
        pos_scores = pos_scores_per_interest.max(dim=1).values
        neg_scores = neg_scores_per_interest.max(dim=1).values

    pos_scores = torch.nan_to_num(pos_scores, nan=0.0, posinf=0.0, neginf=0.0)
    neg_scores = torch.nan_to_num(neg_scores, nan=0.0, posinf=0.0, neginf=0.0)

    rec_loss = bpr_loss(pos_scores, neg_scores)
    div_loss = diversity_regularization(z)

    rec_loss = torch.nan_to_num(rec_loss, nan=0.0, posinf=1e3, neginf=1e3)
    div_loss = torch.nan_to_num(div_loss, nan=0.0, posinf=1e3, neginf=1e3)

    total_loss = rec_loss + diversity_reg_weight * div_loss

    aux = {
        "rec_loss": float(rec_loss.detach().item()),
        "div_loss": float(div_loss.detach().item()),
    }
    return total_loss, aux

In [30]:
@torch.no_grad()
def rerank_candidate_dict_with_model(
    model: nn.Module,
    eval_cases: EvalCases,
    candidate_dict: Dict[int, List[int]],
    k: int,
    eval_batch_size: int,
) -> Dict[int, List[int]]:
    model.eval()
    out: Dict[int, List[int]] = {}

    for start in range(0, len(eval_cases.users), eval_batch_size):
        end = min(start + eval_batch_size, len(eval_cases.users))
        batch_users = eval_cases.users[start:end]
        batch_histories = eval_cases.histories[start:end]

        batch = build_eval_batch_tensors(batch_users, batch_histories)

        candidate_lists = [candidate_dict.get(u, []) for u in batch_users]
        max_cands = max((len(x) for x in candidate_lists), default=0)

        if max_cands == 0:
            for u in batch_users:
                out[u] = []
            continue

        cand_item_ids = torch.zeros((len(batch_users), max_cands), dtype=torch.long, device=DEVICE)
        cand_artist_ids = torch.zeros((len(batch_users), max_cands), dtype=torch.long, device=DEVICE)
        cand_album_ids = torch.zeros((len(batch_users), max_cands), dtype=torch.long, device=DEVICE)
        cand_duration_ids = torch.zeros((len(batch_users), max_cands), dtype=torch.long, device=DEVICE)
        cand_track_title_tokens = torch.zeros(
            (len(batch_users), max_cands, ITEM_TITLE_MAX_TOKENS),
            dtype=torch.long,
            device=DEVICE,
        )
        cand_mask = torch.zeros((len(batch_users), max_cands), dtype=torch.bool, device=DEVICE)

        for i, cand_list in enumerate(candidate_lists):
            for j, item_id in enumerate(cand_list):
                cand_item_ids[i, j] = item_id
                cand_artist_ids[i, j] = itemid_to_artist_id.get(item_id, 0)
                cand_album_ids[i, j] = itemid_to_album_id.get(item_id, 0)
                cand_duration_ids[i, j] = itemid_to_duration_id.get(item_id, 0)
                cand_track_title_tokens[i, j] = torch.tensor(
                    itemid_to_track_title_tokens.get(item_id, [0] * ITEM_TITLE_MAX_TOKENS),
                    dtype=torch.long,
                    device=DEVICE,
                )
                cand_mask[i, j] = True

        if isinstance(model, MultiInterestBase):
            interest_repr = model.get_interest_repr(
                batch["seq_items"],
                batch["seq_artist_ids"],
                batch["seq_album_ids"],
                batch["seq_duration_ids"],
                batch["seq_track_title_tokens"],
                batch["playlist_title_tokens"],
            )
            scores = model.score_items_from_features(
                interest_repr,
                cand_item_ids,
                cand_artist_ids,
                cand_album_ids,
                cand_duration_ids,
                cand_track_title_tokens,
            )
        else:
            user_repr = model.get_user_repr(
                batch["seq_items"],
                batch["seq_artist_ids"],
                batch["seq_album_ids"],
                batch["seq_duration_ids"],
                batch["seq_track_title_tokens"],
                batch["playlist_title_tokens"],
            )
            scores = model.score_items_from_features(
                user_repr,
                cand_item_ids,
                cand_artist_ids,
                cand_album_ids,
                cand_duration_ids,
                cand_track_title_tokens,
            )

        scores = scores.masked_fill(~cand_mask, -1e9)

        topk = min(k, max_cands)
        _, top_idx = torch.topk(scores, k=topk, dim=1)
        top_items = torch.gather(cand_item_ids, 1, top_idx)

        for i, u in enumerate(batch_users):
            out[u] = [int(x) for x in top_items[i].tolist() if int(x) > 0][:k]

    return out

## 12. Evaluation helpers and subset reporting

This section contains:
- subset-based evaluation helpers,
- TopPop full-ranking evaluation,
- and utilities for building per-model subset result tables.

In [31]:
def build_test_subset_rows_for_toppop(
    model_name: str,
    seed,
    eval_cases: EvalCases,
    eval_seqs: Dict[int, List[int]],
    pop_counter: Counter,
    n_items: int,
    subset_masks: Dict[str, np.ndarray],
    k: int = 20,
) -> List[Dict[str, object]]:
    ranked = predict_toppop_ranked(
        eval_cases=eval_cases,
        train_or_eval_seqs=eval_seqs,
        pop_counter=pop_counter,
        n_items=n_items,
        k=k,
    )
    return summarize_subset_metrics_from_ranked(
        ranked=ranked,
        targets=eval_cases.targets,
        n_items=n_items,
        k=k,
        subset_masks=subset_masks,
        model_name=model_name,
        seed=seed,
    )


def build_test_subset_rows_for_repr_model(
    model_name: str,
    seed,
    model: SequenceModelBase,
    eval_cases: EvalCases,
    eval_seqs: Dict[int, List[int]],
    n_items: int,
    subset_masks: Dict[str, np.ndarray],
    k: int = 20,
    eval_batch_size: int = 256,
    item_chunk_size: int = 4096,
) -> List[Dict[str, object]]:
    ranked = predict_repr_model_ranked(
        model=model,
        eval_cases=eval_cases,
        eval_seqs=eval_seqs,
        n_items=n_items,
        k=k,
        eval_batch_size=eval_batch_size,
        item_chunk_size=item_chunk_size,
    )
    return summarize_subset_metrics_from_ranked(
        ranked=ranked,
        targets=eval_cases.targets,
        n_items=n_items,
        k=k,
        subset_masks=subset_masks,
        model_name=model_name,
        seed=seed,
    )


def build_test_subset_rows_for_multi_interest_model(
    model_name: str,
    seed,
    model: MultiInterestBase,
    eval_cases: EvalCases,
    eval_seqs: Dict[int, List[int]],
    n_items: int,
    subset_masks: Dict[str, np.ndarray],
    k: int = 20,
    eval_batch_size: int = 256,
    item_chunk_size: int = 4096,
) -> List[Dict[str, object]]:
    ranked = predict_multi_interest_ranked(
        model=model,
        eval_cases=eval_cases,
        eval_seqs=eval_seqs,
        n_items=n_items,
        k=k,
        eval_batch_size=eval_batch_size,
        item_chunk_size=item_chunk_size,
    )
    return summarize_subset_metrics_from_ranked(
        ranked=ranked,
        targets=eval_cases.targets,
        n_items=n_items,
        k=k,
        subset_masks=subset_masks,
        model_name=model_name,
        seed=seed,
    )

In [32]:
top_r, top_n, top_m, top_cov = evaluate_toppop_full_ranking(
    test_cases,
    test_seqs,
    pop,
    n_items,
    TOPK,
)
print(f"TopPop TEST: R@{TOPK}={top_r:.4f} N@{TOPK}={top_n:.4f} MRR@{TOPK}={top_m:.4f} Coverage@{TOPK}={top_cov:.4f}")

TopPop TEST: R@20=0.0161 N@20=0.0066 MRR@20=0.0039 Coverage@20=0.0030


## 13. Training loops for benchmark models

This section contains the training procedures for:
- GRU4Rec
- SASRec
- SASRec + CL4SRec-style SSL
- SASRec + SimDCL-style SSL
- FMLP-Rec-style efficient baseline
- ComiRec-SA-style SASRec
- ComiRec-SA-style SASRec with label-aware routing and diversity regularization

All models use the same validation protocol and early stopping logic.

In [33]:
def train_gru4rec(train_seqs, valid_cases, test_cases, seed: int):
    set_all_seeds(seed)
    model = GRU4Rec(
        n_items=n_items,
        d_model=D_MODEL,
        hidden_size=GRU_HIDDEN,
        n_layers=1,
        dropout=DROPOUT,
    ).to(DEVICE)

    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    _, dl = make_pairwise_loader(train_seqs, seed)
    stopper = EarlyStopper(patience=PATIENCE, min_delta=1e-4, mode="max")

    for ep in range(EPOCHS):
        model.train()
        total = 0.0

        for batch in dl:
            user_ids = batch["user_id"].to(DEVICE, non_blocking=True)
            seq_items = batch["seq_items"].to(DEVICE, non_blocking=True)
            pos_item = batch["pos_item"].to(DEVICE, non_blocking=True)

            seq_artist_ids = ITEM_ARTIST_IDS_GPU[seq_items]
            seq_album_ids = ITEM_ALBUM_IDS_GPU[seq_items]
            seq_duration_ids = ITEM_DURATION_IDS_GPU[seq_items]
            seq_track_title_tokens = ITEM_TRACK_TITLE_TOKENS_GPU[seq_items]
            playlist_title_tokens = USER_PLAYLIST_TITLE_TOKENS_GPU[user_ids]

            pos_artist_id = ITEM_ARTIST_IDS_GPU[pos_item]
            pos_album_id = ITEM_ALBUM_IDS_GPU[pos_item]
            pos_duration_id = ITEM_DURATION_IDS_GPU[pos_item]
            pos_track_title_tokens = ITEM_TRACK_TITLE_TOKENS_GPU[pos_item]

            neg_items = sample_negatives_on_device(
                seq_items=seq_items,
                pos_item=pos_item,
                n_items=n_items,
                n_neg=30,
            )
            neg_artist_ids = ITEM_ARTIST_IDS_GPU[neg_items]
            neg_album_ids = ITEM_ALBUM_IDS_GPU[neg_items]
            neg_duration_ids = ITEM_DURATION_IDS_GPU[neg_items]
            neg_track_title_tokens = ITEM_TRACK_TITLE_TOKENS_GPU[neg_items]

            u = model.get_user_repr(
                seq_items,
                seq_artist_ids,
                seq_album_ids,
                seq_duration_ids,
                seq_track_title_tokens,
                playlist_title_tokens,
            )

            pos_scores = model.score_items_from_features(
                u,
                pos_item.unsqueeze(1),
                pos_artist_id.unsqueeze(1),
                pos_album_id.unsqueeze(1),
                pos_duration_id.unsqueeze(1),
                pos_track_title_tokens.unsqueeze(1),
            ).squeeze(1)

            neg_scores = model.score_items_from_features(
                u,
                neg_items,
                neg_artist_ids,
                neg_album_ids,
                neg_duration_ids,
                neg_track_title_tokens,
            )

            loss = bpr_loss(pos_scores, neg_scores)

            opt.zero_grad(set_to_none=True)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()

            total += float(loss.item())

        r, n, m, cov = evaluate_repr_model_full_ranking(
            model,
            valid_cases,
            valid_seqs,
            n_items,
            k=TOPK,
            eval_batch_size=EVAL_BATCH_SIZE,
            item_chunk_size=ITEM_CHUNK_SIZE,
        )
        print(
            f"[GRU4Rec][seed={seed}] epoch {ep+1}/{EPOCHS}: "
            f"loss={total/max(1, len(dl)):.4f} | "
            f"valid R@{TOPK}={r:.4f} N@{TOPK}={n:.4f} MRR@{TOPK}={m:.4f} COV@{TOPK}={cov:.4f}"
        )

        if stopper.step(n, model):
            print(f"[GRU4Rec][seed={seed}] early stopping at epoch {ep+1}.")
            break

    stopper.restore_best(model)

    r, n, m, cov = evaluate_repr_model_full_ranking(
        model,
        test_cases,
        test_seqs,
        n_items,
        k=TOPK,
        eval_batch_size=EVAL_BATCH_SIZE,
        item_chunk_size=ITEM_CHUNK_SIZE,
    )

    metrics = {
        "Recall@20": r,
        "NDCG@20": n,
        "MRR@20": m,
        "Coverage@20": cov,
        "best_valid_NDCG@20": float(stopper.best if stopper.best is not None else n),
    }

    subset_rows = build_test_subset_rows_for_repr_model(
        model_name="GRU4Rec",
        seed=seed,
        model=model,
        eval_cases=test_cases,
        eval_seqs=test_seqs,
        n_items=n_items,
        subset_masks=TEST_SUBSET_MASKS,
        k=TOPK,
        eval_batch_size=EVAL_BATCH_SIZE,
        item_chunk_size=ITEM_CHUNK_SIZE,
    )
    return model, metrics, subset_rows


def train_metadata_sasrec(train_seqs, valid_cases, test_cases, seed: int, ssl_mode: Optional[str] = None):
    set_all_seeds(seed)
    model = MetadataSASRec(
        n_items=n_items,
        n_artists=N_ARTISTS,
        n_albums=N_ALBUMS,
        n_durations=N_DURATIONS,
        n_track_title_tokens=N_TRACK_TITLE_TOKENS,
        n_playlist_title_tokens=N_PLAYLIST_TITLE_TOKENS,
        max_len=MAX_SEQ_LEN,
        d_model=D_MODEL,
        n_heads=N_HEADS,
        n_layers=N_LAYERS,
        dropout=DROPOUT,
    ).to(DEVICE)

    tag = {
        None: "MetadataSASRec",
        "cl4srec": "MetadataSASRec+CL4SRec",
        "simdcl": "MetadataSASRec+SimDCL",
    }[ssl_mode]

    if ssl_mode is not None:
        print(f"[{tag}] building SSL loader...")
        _, dl_ssl = make_ssl_loader(train_seqs)
        print(f"[{tag}] SSL loader ready, batches={len(dl_ssl)}")

        ssl_opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

        for ep in range(SSL_EPOCHS):
            model.train()
            total = 0.0

            for batch_idx, (v1, v2) in enumerate(dl_ssl):
                for kk in v1:
                    v1[kk] = v1[kk].to(DEVICE, non_blocking=True)
                    v2[kk] = v2[kk].to(DEVICE, non_blocking=True)

                if ssl_mode == "cl4srec":
                    z1 = model.pooled_ssl_repr(**v1)
                    z2 = model.pooled_ssl_repr(**v2)
                    loss = info_nce_loss(z1, z2, temperature=SSL_TAU)
                elif ssl_mode == "simdcl":
                    z = model.pooled_ssl_repr(**v1)
                    loss = simdcl_loss(z, noise_std=SIMDCL_NOISE_STD, temperature=SSL_TAU)
                else:
                    raise ValueError(ssl_mode)

                ssl_opt.zero_grad(set_to_none=True)
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                ssl_opt.step()

                total += float(loss.item())

            print(f"[{tag} SSL] epoch {ep+1}/{SSL_EPOCHS}: loss={total/max(1, len(dl_ssl)):.4f}")

    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    _, dl = make_pairwise_loader(train_seqs, seed)
    stopper = EarlyStopper(patience=PATIENCE, min_delta=1e-4, mode="max")

    for ep in range(EPOCHS):
        model.train()
        total = 0.0

        for batch in dl:
            user_ids = batch["user_id"].to(DEVICE, non_blocking=True)
            seq_items = batch["seq_items"].to(DEVICE, non_blocking=True)
            pos_item = batch["pos_item"].to(DEVICE, non_blocking=True)

            seq_artist_ids = ITEM_ARTIST_IDS_GPU[seq_items]
            seq_album_ids = ITEM_ALBUM_IDS_GPU[seq_items]
            seq_duration_ids = ITEM_DURATION_IDS_GPU[seq_items]
            seq_track_title_tokens = ITEM_TRACK_TITLE_TOKENS_GPU[seq_items]
            playlist_title_tokens = USER_PLAYLIST_TITLE_TOKENS_GPU[user_ids]

            pos_artist_id = ITEM_ARTIST_IDS_GPU[pos_item]
            pos_album_id = ITEM_ALBUM_IDS_GPU[pos_item]
            pos_duration_id = ITEM_DURATION_IDS_GPU[pos_item]
            pos_track_title_tokens = ITEM_TRACK_TITLE_TOKENS_GPU[pos_item]

            neg_items = sample_negatives_on_device(
                seq_items=seq_items,
                pos_item=pos_item,
                n_items=n_items,
                n_neg=30,
            )
            neg_artist_ids = ITEM_ARTIST_IDS_GPU[neg_items]
            neg_album_ids = ITEM_ALBUM_IDS_GPU[neg_items]
            neg_duration_ids = ITEM_DURATION_IDS_GPU[neg_items]
            neg_track_title_tokens = ITEM_TRACK_TITLE_TOKENS_GPU[neg_items]

            u = model.get_user_repr(
                seq_items,
                seq_artist_ids,
                seq_album_ids,
                seq_duration_ids,
                seq_track_title_tokens,
                playlist_title_tokens,
            )

            pos_scores = model.score_items_from_features(
                u,
                pos_item.unsqueeze(1),
                pos_artist_id.unsqueeze(1),
                pos_album_id.unsqueeze(1),
                pos_duration_id.unsqueeze(1),
                pos_track_title_tokens.unsqueeze(1),
            ).squeeze(1)

            neg_scores = model.score_items_from_features(
                u,
                neg_items,
                neg_artist_ids,
                neg_album_ids,
                neg_duration_ids,
                neg_track_title_tokens,
            )

            loss = bpr_loss(pos_scores, neg_scores)

            opt.zero_grad(set_to_none=True)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()

            total += float(loss.item())

        r, n, m, cov = evaluate_repr_model_full_ranking(
            model,
            valid_cases,
            valid_seqs,
            n_items,
            k=TOPK,
            eval_batch_size=EVAL_BATCH_SIZE,
            item_chunk_size=ITEM_CHUNK_SIZE,
        )

        print(
            f"[{tag}][seed={seed}] epoch {ep+1}/{EPOCHS}: "
            f"loss={total/max(1, len(dl)):.4f} | "
            f"valid R@{TOPK}={r:.4f} N@{TOPK}={n:.4f} MRR@{TOPK}={m:.4f} COV@{TOPK}={cov:.4f}"
        )

        if stopper.step(n, model):
            print(f"[{tag}][seed={seed}] early stopping at epoch {ep+1}.")
            break

    stopper.restore_best(model)

    r, n, m, cov = evaluate_repr_model_full_ranking(
        model,
        test_cases,
        test_seqs,
        n_items,
        k=TOPK,
        eval_batch_size=EVAL_BATCH_SIZE,
        item_chunk_size=ITEM_CHUNK_SIZE,
    )

    metrics = {
        "Recall@20": r,
        "NDCG@20": n,
        "MRR@20": m,
        "Coverage@20": cov,
        "best_valid_NDCG@20": float(stopper.best if stopper.best is not None else n),
    }

    subset_rows = build_test_subset_rows_for_repr_model(
        model_name=tag,
        seed=seed,
        model=model,
        eval_cases=test_cases,
        eval_seqs=test_seqs,
        n_items=n_items,
        subset_masks=TEST_SUBSET_MASKS,
        k=TOPK,
        eval_batch_size=EVAL_BATCH_SIZE,
        item_chunk_size=ITEM_CHUNK_SIZE,
    )
    return model, metrics, subset_rows


def train_fmlprec(train_seqs, valid_cases, test_cases, seed: int):
    set_all_seeds(seed)
    model = FMLPRec(
        n_items=n_items,
        max_len=MAX_SEQ_LEN,
        d_model=D_MODEL,
        n_layers=FMLP_N_LAYERS,
        dropout=FMLP_DROPOUT,
    ).to(DEVICE)

    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    _, dl = make_pairwise_loader(train_seqs, seed)
    stopper = EarlyStopper(patience=PATIENCE, min_delta=1e-4, mode="max")

    for ep in range(EPOCHS):
        model.train()
        total = 0.0

        for batch in dl:
            user_ids = batch["user_id"].to(DEVICE, non_blocking=True)
            seq_items = batch["seq_items"].to(DEVICE, non_blocking=True)
            pos_item = batch["pos_item"].to(DEVICE, non_blocking=True)

            seq_artist_ids = ITEM_ARTIST_IDS_GPU[seq_items]
            seq_album_ids = ITEM_ALBUM_IDS_GPU[seq_items]
            seq_duration_ids = ITEM_DURATION_IDS_GPU[seq_items]
            seq_track_title_tokens = ITEM_TRACK_TITLE_TOKENS_GPU[seq_items]
            playlist_title_tokens = USER_PLAYLIST_TITLE_TOKENS_GPU[user_ids]

            pos_artist_id = ITEM_ARTIST_IDS_GPU[pos_item]
            pos_album_id = ITEM_ALBUM_IDS_GPU[pos_item]
            pos_duration_id = ITEM_DURATION_IDS_GPU[pos_item]
            pos_track_title_tokens = ITEM_TRACK_TITLE_TOKENS_GPU[pos_item]

            neg_items = sample_negatives_on_device(
                seq_items=seq_items,
                pos_item=pos_item,
                n_items=n_items,
                n_neg=30,
            )
            neg_artist_ids = ITEM_ARTIST_IDS_GPU[neg_items]
            neg_album_ids = ITEM_ALBUM_IDS_GPU[neg_items]
            neg_duration_ids = ITEM_DURATION_IDS_GPU[neg_items]
            neg_track_title_tokens = ITEM_TRACK_TITLE_TOKENS_GPU[neg_items]

            u = model.get_user_repr(
                seq_items,
                seq_artist_ids,
                seq_album_ids,
                seq_duration_ids,
                seq_track_title_tokens,
                playlist_title_tokens,
            )

            pos_scores = model.score_items_from_features(
                u,
                pos_item.unsqueeze(1),
                pos_artist_id.unsqueeze(1),
                pos_album_id.unsqueeze(1),
                pos_duration_id.unsqueeze(1),
                pos_track_title_tokens.unsqueeze(1),
            ).squeeze(1)

            neg_scores = model.score_items_from_features(
                u,
                neg_items,
                neg_artist_ids,
                neg_album_ids,
                neg_duration_ids,
                neg_track_title_tokens,
            )

            loss = bpr_loss(pos_scores, neg_scores)

            opt.zero_grad(set_to_none=True)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()

            total += float(loss.item())

        r, n, m, cov = evaluate_repr_model_full_ranking(
            model,
            valid_cases,
            valid_seqs,
            n_items,
            k=TOPK,
            eval_batch_size=EVAL_BATCH_SIZE,
            item_chunk_size=ITEM_CHUNK_SIZE,
        )
        print(
            f"[FMLP-Rec][seed={seed}] epoch {ep+1}/{EPOCHS}: "
            f"loss={total/max(1, len(dl)):.4f} | "
            f"valid R@{TOPK}={r:.4f} N@{TOPK}={n:.4f} MRR@{TOPK}={m:.4f} COV@{TOPK}={cov:.4f}"
        )

        if stopper.step(n, model):
            print(f"[FMLP-Rec][seed={seed}] early stopping at epoch {ep+1}.")
            break

    stopper.restore_best(model)

    r, n, m, cov = evaluate_repr_model_full_ranking(
        model,
        test_cases,
        test_seqs,
        n_items,
        k=TOPK,
        eval_batch_size=EVAL_BATCH_SIZE,
        item_chunk_size=ITEM_CHUNK_SIZE,
    )

    metrics = {
        "Recall@20": r,
        "NDCG@20": n,
        "MRR@20": m,
        "Coverage@20": cov,
        "best_valid_NDCG@20": float(stopper.best if stopper.best is not None else n),
    }

    subset_rows = build_test_subset_rows_for_repr_model(
        model_name="FMLP-Rec",
        seed=seed,
        model=model,
        eval_cases=test_cases,
        eval_seqs=test_seqs,
        n_items=n_items,
        subset_masks=TEST_SUBSET_MASKS,
        k=TOPK,
        eval_batch_size=EVAL_BATCH_SIZE,
        item_chunk_size=ITEM_CHUNK_SIZE,
    )
    return model, metrics, subset_rows


def train_comirec_sa_style(train_seqs, valid_cases, test_cases, seed: int, use_label_aware: bool):
    set_all_seeds(seed)
    model = MetadataComiRecSA(
        n_items=n_items,
        n_artists=N_ARTISTS,
        n_albums=N_ALBUMS,
        n_durations=N_DURATIONS,
        n_track_title_tokens=N_TRACK_TITLE_TOKENS,
        n_playlist_title_tokens=N_PLAYLIST_TITLE_TOKENS,
        max_len=MAX_SEQ_LEN,
        d_model=D_MODEL,
        n_heads=N_HEADS,
        n_layers=N_LAYERS,
        dropout=DROPOUT,
        n_interests=N_INTERESTS,
    ).to(DEVICE)

    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    _, dl = make_pairwise_loader(train_seqs, seed)
    stopper = EarlyStopper(patience=PATIENCE, min_delta=1e-4, mode="max")

    tag = "MetadataComiRecSA" if not use_label_aware else "MetadataComiRecSA+LabelAware+DivReg"

    for ep in range(EPOCHS):
        model.train()
        total = 0.0
        total_rec = 0.0
        total_div = 0.0

        for batch in dl:
            user_ids = batch["user_id"].to(DEVICE, non_blocking=True)
            seq_items = batch["seq_items"].to(DEVICE, non_blocking=True)
            pos_item = batch["pos_item"].to(DEVICE, non_blocking=True)

            seq_artist_ids = ITEM_ARTIST_IDS_GPU[seq_items]
            seq_album_ids = ITEM_ALBUM_IDS_GPU[seq_items]
            seq_duration_ids = ITEM_DURATION_IDS_GPU[seq_items]
            seq_track_title_tokens = ITEM_TRACK_TITLE_TOKENS_GPU[seq_items]
            playlist_title_tokens = USER_PLAYLIST_TITLE_TOKENS_GPU[user_ids]

            pos_artist_id = ITEM_ARTIST_IDS_GPU[pos_item]
            pos_album_id = ITEM_ALBUM_IDS_GPU[pos_item]
            pos_duration_id = ITEM_DURATION_IDS_GPU[pos_item]
            pos_track_title_tokens = ITEM_TRACK_TITLE_TOKENS_GPU[pos_item]

            neg_items = sample_negatives_on_device(
                seq_items=seq_items,
                pos_item=pos_item,
                n_items=n_items,
                n_neg=30,
            )
            neg_artist_ids = ITEM_ARTIST_IDS_GPU[neg_items]
            neg_album_ids = ITEM_ALBUM_IDS_GPU[neg_items]
            neg_duration_ids = ITEM_DURATION_IDS_GPU[neg_items]
            neg_track_title_tokens = ITEM_TRACK_TITLE_TOKENS_GPU[neg_items]

            batch_full = {
                "seq_items": seq_items,
                "seq_artist_ids": seq_artist_ids,
                "seq_album_ids": seq_album_ids,
                "seq_duration_ids": seq_duration_ids,
                "seq_track_title_tokens": seq_track_title_tokens,
                "playlist_title_tokens": playlist_title_tokens,
                "pos_item": pos_item,
                "pos_artist_id": pos_artist_id,
                "pos_album_id": pos_album_id,
                "pos_duration_id": pos_duration_id,
                "pos_track_title_tokens": pos_track_title_tokens,
                "neg_items": neg_items,
                "neg_artist_ids": neg_artist_ids,
                "neg_album_ids": neg_album_ids,
                "neg_duration_ids": neg_duration_ids,
                "neg_track_title_tokens": neg_track_title_tokens,
            }

            loss, aux = label_aware_comirec_loss(
                model=model,
                batch=batch_full,
                label_tau=COMIREC_LABEL_TAU,
                diversity_reg_weight=COMIREC_DIVERSITY_REG,
                use_label_aware=use_label_aware,
            )

            opt.zero_grad(set_to_none=True)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()

            total += float(loss.item())
            total_rec += aux["rec_loss"]
            total_div += aux["div_loss"]

        r, n, m, cov = evaluate_multi_interest_full_ranking(
            model,
            valid_cases,
            valid_seqs,
            n_items,
            k=TOPK,
            eval_batch_size=EVAL_BATCH_SIZE,
            item_chunk_size=ITEM_CHUNK_SIZE,
        )
        print(
            f"[{tag}][seed={seed}] epoch {ep+1}/{EPOCHS}: "
            f"loss={total/max(1, len(dl)):.4f} "
            f"rec={total_rec/max(1, len(dl)):.4f} "
            f"div={total_div/max(1, len(dl)):.4f} | "
            f"valid R@{TOPK}={r:.4f} N@{TOPK}={n:.4f} MRR@{TOPK}={m:.4f} COV@{TOPK}={cov:.4f}"
        )

        if stopper.step(n, model):
            print(f"[{tag}][seed={seed}] early stopping at epoch {ep+1}.")
            break

    stopper.restore_best(model)

    r, n, m, cov = evaluate_multi_interest_full_ranking(
        model,
        test_cases,
        test_seqs,
        n_items,
        k=TOPK,
        eval_batch_size=EVAL_BATCH_SIZE,
        item_chunk_size=ITEM_CHUNK_SIZE,
    )

    metrics = {
        "Recall@20": r,
        "NDCG@20": n,
        "MRR@20": m,
        "Coverage@20": cov,
        "best_valid_NDCG@20": float(stopper.best if stopper.best is not None else n),
        **analyze_interest_usage(model, test_cases),
    }

    subset_rows = build_test_subset_rows_for_multi_interest_model(
        model_name=tag,
        seed=seed,
        model=model,
        eval_cases=test_cases,
        eval_seqs=test_seqs,
        n_items=n_items,
        subset_masks=TEST_SUBSET_MASKS,
        k=TOPK,
        eval_batch_size=EVAL_BATCH_SIZE,
        item_chunk_size=ITEM_CHUNK_SIZE,
    )
    return model, metrics, subset_rows

## 13.1 Checkpointing and IRRT model-loading helpers

This section defines:
- checkpoint naming and saving
- model reconstruction from model names
- IRRT evaluation over saved checkpoints

The IRRT experiments reuse the exact trained model checkpoints from the main benchmark runs.

In [34]:
def slugify_model_name(name: str) -> str:
    return (
        name.replace("+", "_plus_")
        .replace("/", "_")
        .replace(" ", "_")
        .replace("-", "_")
    )


def build_model_from_name(model_name: str, config: Dict[str, object]) -> nn.Module:
    if model_name == "GRU4Rec":
        return GRU4Rec(
            n_items=n_items,
            d_model=int(config["D_MODEL"]),
            hidden_size=int(config["GRU_HIDDEN"]),
            n_layers=2,
            dropout=float(config["DROPOUT"]),
        )

    if model_name in {"MetadataSASRec", "MetadataSASRec+CL4SRec", "MetadataSASRec+SimDCL"}:
        return MetadataSASRec(
            n_items=n_items,
            n_artists=N_ARTISTS,
            n_albums=N_ALBUMS,
            n_durations=N_DURATIONS,
            n_track_title_tokens=N_TRACK_TITLE_TOKENS,
            n_playlist_title_tokens=N_PLAYLIST_TITLE_TOKENS,
            max_len=int(config["MAX_SEQ_LEN"]),
            d_model=int(config["D_MODEL"]),
            n_heads=int(config["N_HEADS"]),
            n_layers=int(config["N_LAYERS"]),
            dropout=float(config["DROPOUT"]),
        )

    if model_name == "FMLP-Rec":
        return FMLPRec(
            n_items=n_items,
            max_len=int(config["MAX_SEQ_LEN"]),
            d_model=int(config["D_MODEL"]),
            n_layers=int(config["FMLP_N_LAYERS"]),
            dropout=float(config["FMLP_DROPOUT"]),
        )

    if model_name in {"MetadataComiRecSA", "MetadataComiRecSA+LabelAware+DivReg"}:
        return MetadataComiRecSA(
            n_items=n_items,
            n_artists=N_ARTISTS,
            n_albums=N_ALBUMS,
            n_durations=N_DURATIONS,
            n_track_title_tokens=N_TRACK_TITLE_TOKENS,
            n_playlist_title_tokens=N_PLAYLIST_TITLE_TOKENS,
            max_len=int(config["MAX_SEQ_LEN"]),
            d_model=int(config["D_MODEL"]),
            n_heads=int(config["N_HEADS"]),
            n_layers=int(config["N_LAYERS"]),
            dropout=float(config["DROPOUT"]),
            n_interests=int(config["N_INTERESTS"]),
        )

    raise ValueError(f"Unsupported model_name for checkpoint loading: {model_name}")


def save_model_checkpoint(model: nn.Module, model_name: str, seed, checkpoint_path: Path) -> None:
    payload = {
        "model_name": model_name,
        "seed": seed,
        "config": dict(EXPERIMENT_CONFIG),
        "state_dict": model.state_dict(),
    }
    torch.save(payload, checkpoint_path)


def load_model_checkpoint(model_name: str, checkpoint_path: str) -> nn.Module:
    payload = torch.load(checkpoint_path, map_location=DEVICE)
    config = payload.get("config", dict(EXPERIMENT_CONFIG))
    state_dict = payload["state_dict"] if "state_dict" in payload else payload

    model = build_model_from_name(model_name, config).to(DEVICE)
    model.load_state_dict(state_dict)
    model.eval()
    return model


def run_irrt_retrieval_for_checkpoint_row(
    row: pd.Series,
    eval_cases: EvalCases,
    dense_candidates: Dict[int, List[int]],
    sparse_candidates: Dict[int, List[int]],
    split_name: str,
) -> List[Dict[str, object]]:
    model_name = row["model"]
    seed = row["seed"]

    hybrid_candidates = build_hybrid_candidate_dict(
        dense_dict=dense_candidates,
        sparse_dict=sparse_candidates,
        dense_weight=IRRT_HYBRID_DENSE_WEIGHT,
        sparse_weight=IRRT_HYBRID_SPARSE_WEIGHT,
        top_m=max(IRRT_RETRIEVAL_CUTOFFS),
    )

    retrieval_rows = []
    for pipeline_name, cand_dict in [
        ("SparseLexical", sparse_candidates),
        (EXACT_DENSE_PIPELINE_NAME, dense_candidates),
        (HYBRID_PIPELINE_NAME, hybrid_candidates),
    ]:
        rec_row = {
            "split": split_name,
            "model": model_name,
            "seed": seed,
            "pipeline": pipeline_name,
        }
        for m in IRRT_RETRIEVAL_CUTOFFS:
            rec_row[f"CandidateRecall@{m}"] = candidate_recall_at_m(
                cand_dict,
                eval_cases,
                m,
            )
        retrieval_rows.append(rec_row)

    return retrieval_rows

## 14. Main recommendation benchmark run

This section runs the main recommendation benchmark across:
- classic baselines
- metadata-aware sequential models
- SSL-enhanced metadata-aware models
- an efficient sequential alternative
- metadata-aware multi-interest models

For every neural run, the notebook also saves the trained checkpoint.
These checkpoints are later reused in the IRRT experiments.

In [35]:
results = []
subset_results = []

top_valid_r, top_valid_n, top_valid_m, top_valid_cov = evaluate_toppop_full_ranking(
    valid_cases,
    valid_seqs,
    pop,
    n_items,
    TOPK,
)

experiment_plan = [
    ("TopPop", "fixed", lambda: (
        None,
        {
            "Recall@20": top_r,
            "NDCG@20": top_n,
            "MRR@20": top_m,
            "Coverage@20": top_cov,
            "best_valid_NDCG@20": top_valid_n,
        },
        build_test_subset_rows_for_toppop(
            model_name="TopPop",
            seed="fixed",
            eval_cases=test_cases,
            eval_seqs=test_seqs,
            pop_counter=pop,
            n_items=n_items,
            subset_masks=TEST_SUBSET_MASKS,
            k=TOPK,
        ),
    ))
]

# for seed in SEEDS:
#     experiment_plan.append(
#         ("GRU4Rec", seed, lambda seed=seed: train_gru4rec(train_seqs, valid_cases, test_cases, seed))
#     )

# for seed in SEEDS:
#     experiment_plan.append(
#         ("MetadataSASRec", seed, lambda seed=seed: train_metadata_sasrec(
#             train_seqs, valid_cases, test_cases, seed, ssl_mode=None
#         ))
#     )

# if DO_CL4SREC_SSL:
#     for seed in SEEDS:
#         experiment_plan.append(
#             ("MetadataSASRec+CL4SRec", seed, lambda seed=seed: train_metadata_sasrec(
#                 train_seqs, valid_cases, test_cases, seed, ssl_mode="cl4srec"
#             ))
#         )

# if DO_SIMDCL:
#     for seed in SEEDS:
#         experiment_plan.append(
#             ("MetadataSASRec+SimDCL", seed, lambda seed=seed: train_metadata_sasrec(
#                 train_seqs, valid_cases, test_cases, seed, ssl_mode="simdcl"
#             ))
#         )

for seed in SEEDS:
    experiment_plan.append(
        ("FMLP-Rec", seed, lambda seed=seed: train_fmlprec(train_seqs, valid_cases, test_cases, seed))
    )

for seed in SEEDS:
    experiment_plan.append(
        ("MetadataComiRecSA", seed, lambda seed=seed: train_comirec_sa_style(
            train_seqs, valid_cases, test_cases, seed, use_label_aware=False
        ))
    )

for seed in SEEDS:
    experiment_plan.append(
        ("MetadataComiRecSA+LabelAware+DivReg", seed, lambda seed=seed: train_comirec_sa_style(
            train_seqs, valid_cases, test_cases, seed, use_label_aware=True
        ))
    )

print(f"Total runs: {len(experiment_plan)}")

for run_idx, (model_name, seed, run_fn) in enumerate(tqdm(experiment_plan, desc="Experiments"), start=1):
    print("=" * 80)
    print(f"START [{run_idx}/{len(experiment_plan)}] model={model_name}, seed={seed}")

    t0 = perf_counter()
    model_obj, metrics, subset_rows = run_fn()
    elapsed = perf_counter() - t0

    checkpoint_path = ""
    if model_obj is not None:
        ckpt_name = f"{slugify_model_name(model_name)}__seed_{seed}.pt"
        ckpt_path_obj = MODEL_DIR / ckpt_name
        save_model_checkpoint(model_obj, model_name=model_name, seed=seed, checkpoint_path=ckpt_path_obj)
        checkpoint_path = str(ckpt_path_obj)

    row = {
        "model": model_name,
        "seed": seed,
        "runtime_sec": round(elapsed, 2),
        "checkpoint_path": checkpoint_path,
        **metrics,
    }
    results.append(row)
    subset_results.extend(subset_rows)

    results_df = pd.DataFrame(results)
    subset_results_df = pd.DataFrame(subset_results)

    results_df.to_csv(PROJECT_DIR / "results_full_ranking_checkpoint.csv", index=False)
    subset_results_df.to_csv(PROJECT_DIR / "results_subset_checkpoint.csv", index=False)

    print(f"DONE  [{run_idx}/{len(experiment_plan)}] model={model_name}, seed={seed}, time={elapsed:.2f}s")

    clear_output(wait=True)
    print(f"Completed runs: {run_idx}/{len(experiment_plan)}")
    display(results_df.sort_values(["model", "seed"], kind="stable").reset_index(drop=True))

results_df = pd.DataFrame(results).sort_values(["model", "seed"], kind="stable").reset_index(drop=True)
subset_results_df = pd.DataFrame(subset_results).sort_values(["model", "subset", "seed"], kind="stable").reset_index(drop=True)

display(results_df)
display(subset_results_df.head(30))

Completed runs: 10/10


,model,seed,runtime_sec,checkpoint_path,Recall@20,NDCG@20,MRR@20,Coverage@20,best_valid_NDCG@20,interest_usage_entropy,interest_usage_max_share,interest_target_score_entropy_mean
0,FMLP-Rec,42,2011.66,C:\Users\sokos\DataspellProjects\MML_Project\s...,0.020739,0.008284,0.004915,0.14235,0.009212,NaN,NaN,NaN
1,FMLP-Rec,43,2021.03,C:\Users\sokos\DataspellProjects\MML_Project\s...,0.023680,0.009727,0.005915,0.17035,0.010480,NaN,NaN,NaN
2,FMLP-Rec,44,2035.61,C:\Users\sokos\DataspellProjects\MML_Project\s...,0.019892,0.007908,0.004651,0.15455,0.008948,NaN,NaN,NaN
3,MetadataComiRecSA,42,6816.47,C:\Users\sokos\DataspellProjects\MML_Project\s...,0.071659,0.027782,0.015902,0.50265,0.029543,1.045301,0.477334,0.681115
4,MetadataComiRecSA,43,6714.41,C:\Users\sokos\DataspellProjects\MML_Project\s...,0.073309,0.028740,0.016657,0.50015,0.030638,1.017841,0.433482,0.653242
5,MetadataComiRecSA,44,6733.66,C:\Users\sokos\DataspellProjects\MML_Project\s...,0.079994,0.031972,0.018911,0.50380,0.033597,0.720720,0.516768,0.541604
6,MetadataComiRecSA+LabelAware+DivReg,42,5362.31,C:\Users\sokos\DataspellProjects\MML_Project\s...,0.000946,0.000352,0.000193,0.08860,0.000344,1.313754,0.354570,0.783166
7,MetadataComiRecSA+LabelAware+DivReg,43,5368.91,C:\Users\sokos\DataspellProjects\MML_Project\s...,0.025282,0.009666,0.005459,0.25235,0.010053,1.327533,0.355550,0.782603
8,MetadataComiRecSA+LabelAware+DivReg,44,5370.32,C:\Users\sokos\DataspellProjects\MML_Project\s...,0.012549,0.004691,0.002584,0.17520,0.004985,1.378631,0.281514,0.591536
9,TopPop,fixed,4.35,,0.016076,0.006562,0.003925,0.00300,0.006294,NaN,NaN,NaN


,model,seed,runtime_sec,checkpoint_path,Recall@20,NDCG@20,MRR@20,Coverage@20,best_valid_NDCG@20,interest_usage_entropy,interest_usage_max_share,interest_target_score_entropy_mean
0,FMLP-Rec,42,2011.66,C:\Users\sokos\DataspellProjects\MML_Project\s...,0.020739,0.008284,0.004915,0.14235,0.009212,NaN,NaN,NaN
1,FMLP-Rec,43,2021.03,C:\Users\sokos\DataspellProjects\MML_Project\s...,0.023680,0.009727,0.005915,0.17035,0.010480,NaN,NaN,NaN
2,FMLP-Rec,44,2035.61,C:\Users\sokos\DataspellProjects\MML_Project\s...,0.019892,0.007908,0.004651,0.15455,0.008948,NaN,NaN,NaN
3,MetadataComiRecSA,42,6816.47,C:\Users\sokos\DataspellProjects\MML_Project\s...,0.071659,0.027782,0.015902,0.50265,0.029543,1.045301,0.477334,0.681115
4,MetadataComiRecSA,43,6714.41,C:\Users\sokos\DataspellProjects\MML_Project\s...,0.073309,0.028740,0.016657,0.50015,0.030638,1.017841,0.433482,0.653242
5,MetadataComiRecSA,44,6733.66,C:\Users\sokos\DataspellProjects\MML_Project\s...,0.079994,0.031972,0.018911,0.50380,0.033597,0.720720,0.516768,0.541604
6,MetadataComiRecSA+LabelAware+DivReg,42,5362.31,C:\Users\sokos\DataspellProjects\MML_Project\s...,0.000946,0.000352,0.000193,0.08860,0.000344,1.313754,0.354570,0.783166
7,MetadataComiRecSA+LabelAware+DivReg,43,5368.91,C:\Users\sokos\DataspellProjects\MML_Project\s...,0.025282,0.009666,0.005459,0.25235,0.010053,1.327533,0.355550,0.782603
8,MetadataComiRecSA+LabelAware+DivReg,44,5370.32,C:\Users\sokos\DataspellProjects\MML_Project\s...,0.012549,0.004691,0.002584,0.17520,0.004985,1.378631,0.281514,0.591536
9,TopPop,fixed,4.35,,0.016076,0.006562,0.003925,0.00300,0.006294,NaN,NaN,NaN


,model,seed,subset,n_cases,Recall@20,NDCG@20,MRR@20,Coverage@20
0,FMLP-Rec,42,all,558270,0.020739,0.008284,0.004915,0.14235
1,FMLP-Rec,43,all,558270,0.023680,0.009727,0.005915,0.17035
2,FMLP-Rec,44,all,558270,0.019892,0.007908,0.004651,0.15455
3,FMLP-Rec,42,div_heterogeneous,184229,0.016501,0.006400,0.003681,0.12220
4,FMLP-Rec,43,div_heterogeneous,184229,0.019986,0.008565,0.005453,0.14605
5,FMLP-Rec,44,div_heterogeneous,184229,0.015421,0.005996,0.003441,0.13445
6,FMLP-Rec,42,div_homogeneous,184229,0.023183,0.009432,0.005698,0.12940
7,FMLP-Rec,43,div_homogeneous,184229,0.025734,0.010557,0.006399,0.15375
8,FMLP-Rec,44,div_homogeneous,184229,0.023237,0.009476,0.005719,0.13930
9,FMLP-Rec,42,div_medium,189812,0.022480,0.008998,0.005353,0.12620


## 14.1 IRRT evaluation over saved checkpoints

This section runs the IRRT experiments on the saved neural checkpoints.

For every trained checkpoint, the notebook evaluates:
- sparse lexical retrieval
- dense retrieval from learned representations
- hybrid sparse+dense retrieval with reciprocal-rank fusion
- reranking of sparse, dense, and hybrid shortlists

The evaluation is computed on both validation and test splits, and then aggregated across seeds.
This avoids test-time model selection leakage.

In [36]:
irrt_retrieval_rows = []

neural_results_df = results_df[
    results_df["checkpoint_path"].fillna("") != ""
].copy().reset_index(drop=True)

top_m_max = max(IRRT_RETRIEVAL_CUTOFFS)

print("Precomputing model-independent sparse retrieval...")
valid_sparse_candidates = build_sparse_candidate_dict(
    eval_cases=valid_cases,
    eval_seqs=valid_seqs,
    top_m=top_m_max,
    title_weight=IRRT_TITLE_QUERY_WEIGHT,
    history_weight=IRRT_HISTORY_QUERY_WEIGHT,
    history_last_n=IRRT_HISTORY_TEXT_LAST_N,
)
test_sparse_candidates = build_sparse_candidate_dict(
    eval_cases=test_cases,
    eval_seqs=test_seqs,
    top_m=top_m_max,
    title_weight=IRRT_TITLE_QUERY_WEIGHT,
    history_weight=IRRT_HISTORY_QUERY_WEIGHT,
    history_last_n=IRRT_HISTORY_TEXT_LAST_N,
)

sparse_retrieval_rows = []
for split_name, eval_cases_obj, sparse_dict in [
    ("valid", valid_cases, valid_sparse_candidates),
    ("test", test_cases, test_sparse_candidates),
]:
    row = {
        "split": split_name,
        "pipeline": "SparseLexical",
    }
    for m in IRRT_RETRIEVAL_CUTOFFS:
        row[f"CandidateRecall@{m}"] = candidate_recall_at_m(
            sparse_dict,
            eval_cases_obj,
            m,
        )
    sparse_retrieval_rows.append(row)

sparse_retrieval_df = pd.DataFrame(sparse_retrieval_rows)
display(sparse_retrieval_df)

for _, row in tqdm(
    neural_results_df.iterrows(),
    total=len(neural_results_df),
    desc="IRRT retrieval over checkpoints",
):
    model = load_model_checkpoint(row["model"], row["checkpoint_path"])

    valid_dense_candidates = build_dense_candidate_dict(
        model=model,
        eval_cases=valid_cases,
        eval_seqs=valid_seqs,
        top_m=top_m_max,
        eval_batch_size=EVAL_BATCH_SIZE,
        item_chunk_size=ITEM_CHUNK_SIZE,
        cache_key=row["checkpoint_path"],
    )
    test_dense_candidates = build_dense_candidate_dict(
        model=model,
        eval_cases=test_cases,
        eval_seqs=test_seqs,
        top_m=top_m_max,
        eval_batch_size=EVAL_BATCH_SIZE,
        item_chunk_size=ITEM_CHUNK_SIZE,
        cache_key=row["checkpoint_path"],
    )

    valid_ret_rows = run_irrt_retrieval_for_checkpoint_row(
        row=row,
        eval_cases=valid_cases,
        dense_candidates=valid_dense_candidates,
        sparse_candidates=valid_sparse_candidates,
        split_name="valid",
    )
    test_ret_rows = run_irrt_retrieval_for_checkpoint_row(
        row=row,
        eval_cases=test_cases,
        dense_candidates=test_dense_candidates,
        sparse_candidates=test_sparse_candidates,
        split_name="test",
    )

    irrt_retrieval_rows.extend(valid_ret_rows)
    irrt_retrieval_rows.extend(test_ret_rows)

    cache_prefix = f"{row['checkpoint_path']}::items::"
    for kk in list(ITEM_MATRIX_CACHE.keys()):
        if kk.startswith(cache_prefix):
            del ITEM_MATRIX_CACHE[kk]

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

irrt_retrieval_df = pd.DataFrame(irrt_retrieval_rows).sort_values(
    ["split", "pipeline", "model", "seed"],
    kind="stable",
).reset_index(drop=True)

sparse_retrieval_df.to_csv(PROJECT_DIR / "irrt_sparse_retrieval_results.csv", index=False)
irrt_retrieval_df.to_csv(PROJECT_DIR / "irrt_retrieval_results.csv", index=False)

display(sparse_retrieval_df)
display(irrt_retrieval_df.head(30))

Precomputing model-independent sparse retrieval...


,split,pipeline,CandidateRecall@50,CandidateRecall@100,CandidateRecall@200
0,valid,SparseLexical,0.123650,0.159626,0.197422
1,test,SparseLexical,0.108293,0.141817,0.177862


IRRT retrieval over checkpoints:   0%|          | 0/9 [00:00<?, ?it/s]

,split,pipeline,CandidateRecall@50,CandidateRecall@100,CandidateRecall@200
0,valid,SparseLexical,0.123650,0.159626,0.197422
1,test,SparseLexical,0.108293,0.141817,0.177862


,split,model,seed,pipeline,CandidateRecall@50,CandidateRecall@100,CandidateRecall@200
0,test,FMLP-Rec,42,ExactDenseRetriever,0.041315,0.068657,0.111271
1,test,FMLP-Rec,43,ExactDenseRetriever,0.044392,0.070201,0.110362
2,test,FMLP-Rec,44,ExactDenseRetriever,0.038621,0.064214,0.104020
3,test,MetadataComiRecSA,42,ExactDenseRetriever,0.156886,0.241693,0.353179
4,test,MetadataComiRecSA,43,ExactDenseRetriever,0.159383,0.243420,0.355609
5,test,MetadataComiRecSA,44,ExactDenseRetriever,0.167872,0.253342,0.367365
6,test,MetadataComiRecSA+LabelAware+DivReg,42,ExactDenseRetriever,0.027317,0.047631,0.082879
7,test,MetadataComiRecSA+LabelAware+DivReg,43,ExactDenseRetriever,0.053483,0.088959,0.143536
8,test,MetadataComiRecSA+LabelAware+DivReg,44,ExactDenseRetriever,0.027879,0.048228,0.079852
9,test,FMLP-Rec,42,HybridRRF,0.103174,0.146916,0.201648


## 15. Aggregate recommendation and IRRT results across seeds

After all runs finish, I summarize:
- full-ranking recommendation metrics
- subset metrics
- IRRT retrieval candidate recall
- IRRT reranked shortlist quality
- interest-usage statistics

All neural results are aggregated across seeds with mean values and confidence intervals.

In [37]:
def summarize_metric(x: pd.Series) -> pd.Series:
    x = x.astype(float)
    n = len(x)
    mean = x.mean()
    std = x.std(ddof=1) if n > 1 else 0.0
    ci95 = 1.96 * std / math.sqrt(n) if n > 1 else 0.0
    return pd.Series({"mean": mean, "std": std, "ci95": ci95, "n_seeds": n})


main_metric_cols = ["Recall@20", "NDCG@20", "MRR@20", "Coverage@20"]
interest_metric_cols = [
    "interest_usage_entropy",
    "interest_usage_max_share",
    "interest_target_score_entropy_mean",
]
summary_rows = []

for model_name, g in results_df.groupby("model", sort=False):
    row = {"model": model_name}

    if model_name == "TopPop":
        for col in main_metric_cols:
            row[col] = f"{float(g[col].iloc[0]):.4f}"
        row["best_valid_NDCG@20"] = f"{float(g['best_valid_NDCG@20'].iloc[0]):.4f}"
        row["n_seeds"] = 1
    else:
        for col in main_metric_cols:
            s = summarize_metric(g[col])
            row[col] = f"{s['mean']:.4f} ± {s['ci95']:.4f}"
        
        for col in interest_metric_cols:
            if col in g.columns:
                vals = g[col].dropna()
                if len(vals) == 0:
                    continue
                if len(vals) == 1:
                    row[col] = f"{float(vals.iloc[0]):.4f}"
                else:
                    s = summarize_metric(vals)
                    row[col] = f"{s['mean']:.4f} ± {s['ci95']:.4f}"

        s_valid = summarize_metric(g["best_valid_NDCG@20"])
        row["best_valid_NDCG@20"] = f"{s_valid['mean']:.4f} ± {s_valid['ci95']:.4f}"
        row["n_seeds"] = int(s_valid["n_seeds"])

    summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows)
display(summary_df)

,model,Recall@20,NDCG@20,MRR@20,Coverage@20,best_valid_NDCG@20,n_seeds,interest_usage_entropy,interest_usage_max_share,interest_target_score_entropy_mean
0,FMLP-Rec,0.0214 ± 0.0023,0.0086 ± 0.0011,0.0052 ± 0.0008,0.1557 ± 0.0159,0.0095 ± 0.0009,3,NaN,NaN,NaN
1,MetadataComiRecSA,0.0750 ± 0.0050,0.0295 ± 0.0025,0.0172 ± 0.0018,0.5022 ± 0.0021,0.0313 ± 0.0024,3,0.9280 ± 0.2037,0.4759 ± 0.0471,0.6253 ± 0.0835
2,MetadataComiRecSA+LabelAware+DivReg,0.0129 ± 0.0138,0.0049 ± 0.0053,0.0027 ± 0.0030,0.1721 ± 0.0927,0.0051 ± 0.0055,3,1.3400 ± 0.0387,0.3305 ± 0.0481,0.7191 ± 0.1250
3,TopPop,0.0161,0.0066,0.0039,0.0030,0.0063,1,NaN,NaN,NaN


In [38]:
subset_summary_rows = []

for (model_name, subset_name), g in subset_results_df.groupby(["model", "subset"], sort=False):
    row = {
        "model": model_name,
        "subset": subset_name,
        "n_cases": int(g["n_cases"].iloc[0]),
    }

    for col in [f"Recall@{TOPK}", f"NDCG@{TOPK}", f"MRR@{TOPK}", f"Coverage@{TOPK}"]:
        vals = g[col].astype(float)
        if len(vals) == 1:
            row[col] = f"{float(vals.iloc[0]):.4f}"
        else:
            s = summarize_metric(vals)
            row[col] = f"{s['mean']:.4f} ± {s['ci95']:.4f}"

    subset_summary_rows.append(row)

subset_summary_df = pd.DataFrame(subset_summary_rows)
display(subset_summary_df.sort_values(["subset", "model"]).reset_index(drop=True))

,model,subset,n_cases,Recall@20,NDCG@20,MRR@20,Coverage@20
0,FMLP-Rec,all,558270,0.0214 ± 0.0023,0.0086 ± 0.0011,0.0052 ± 0.0008,0.1557 ± 0.0159
1,MetadataComiRecSA,all,558270,0.0750 ± 0.0050,0.0295 ± 0.0025,0.0172 ± 0.0018,0.5022 ± 0.0021
2,MetadataComiRecSA+LabelAware+DivReg,all,558270,0.0129 ± 0.0138,0.0049 ± 0.0053,0.0027 ± 0.0030,0.1721 ± 0.0927
3,TopPop,all,558270,0.0161,0.0066,0.0039,0.0030
4,FMLP-Rec,div_heterogeneous,184229,0.0173 ± 0.0027,0.0070 ± 0.0016,0.0042 ± 0.0012,0.1342 ± 0.0135
5,MetadataComiRecSA,div_heterogeneous,184229,0.0639 ± 0.0056,0.0253 ± 0.0028,0.0148 ± 0.0020,0.4410 ± 0.0019
6,MetadataComiRecSA+LabelAware+DivReg,div_heterogeneous,184229,0.0106 ± 0.0107,0.0040 ± 0.0041,0.0022 ± 0.0023,0.1531 ± 0.0837
7,TopPop,div_heterogeneous,184229,0.0127,0.0051,0.0030,0.0024
8,FMLP-Rec,div_homogeneous,184229,0.0241 ± 0.0016,0.0098 ± 0.0007,0.0059 ± 0.0005,0.1408 ± 0.0139
9,MetadataComiRecSA,div_homogeneous,184229,0.0862 ± 0.0047,0.0338 ± 0.0022,0.0196 ± 0.0015,0.4402 ± 0.0073


In [39]:
subset_recall_pivot = subset_results_df.pivot_table(
    index="subset",
    columns="model",
    values=f"Recall@{TOPK}",
    aggfunc="mean",
)

subset_ndcg_pivot = subset_results_df.pivot_table(
    index="subset",
    columns="model",
    values=f"NDCG@{TOPK}",
    aggfunc="mean",
)

subset_mrr_pivot = subset_results_df.pivot_table(
    index="subset",
    columns="model",
    values=f"MRR@{TOPK}",
    aggfunc="mean",
)

display(subset_recall_pivot)
display(subset_ndcg_pivot)
display(subset_mrr_pivot)

model,FMLP-Rec,MetadataComiRecSA,MetadataComiRecSA+LabelAware+DivReg,TopPop
subset,,,,
all,0.021437,0.074987,0.012926,0.016076
div_heterogeneous,0.017303,0.063875,0.010590,0.012712
div_homogeneous,0.024052,0.086237,0.016508,0.016431
div_medium,0.022912,0.074853,0.011715,0.018998
len_long,0.021999,0.067151,0.010863,0.018237
len_medium,0.021448,0.076600,0.013381,0.016561
len_short,0.020869,0.081206,0.014534,0.013466
target_pop_head,0.038539,0.120814,0.019831,0.029978
target_pop_mid,0.002001,0.025047,0.005628,0.000000


model,FMLP-Rec,MetadataComiRecSA,MetadataComiRecSA+LabelAware+DivReg,TopPop
subset,,,,
all,0.008640,0.029498,0.004903,0.006562
div_heterogeneous,0.006987,0.025270,0.004011,0.005140
div_homogeneous,0.009822,0.033778,0.006347,0.006691
div_medium,0.009096,0.029448,0.004366,0.007818
len_long,0.008567,0.026124,0.003959,0.007525
len_medium,0.008617,0.030159,0.005088,0.006791
len_short,0.008733,0.032208,0.005662,0.005385
target_pop_head,0.015541,0.048490,0.007648,0.012237
target_pop_mid,0.000795,0.008604,0.001975,0.000000


model,FMLP-Rec,MetadataComiRecSA,MetadataComiRecSA+LabelAware+DivReg,TopPop
subset,,,,
all,0.005160,0.017157,0.002745,0.003925
div_heterogeneous,0.004192,0.014792,0.002246,0.003049
div_homogeneous,0.005939,0.019552,0.003605,0.003993
div_medium,0.005345,0.017127,0.002396,0.004708
len_long,0.004922,0.015004,0.002109,0.004545
len_medium,0.005134,0.017554,0.002865,0.004083
len_short,0.005423,0.018909,0.003262,0.003156
target_pop_head,0.009288,0.028783,0.004364,0.007319
target_pop_mid,0.000468,0.004255,0.001002,0.000000


## 15.1 Aggregate IRRT retrieval results

This section summarizes retrieval candidate recall for:
- sparse lexical retrieval
- dense retrieval
- hybrid sparse+dense retrieval
across validation and test splits.

In [40]:
print("Model-independent sparse retrieval summary:")
display(sparse_retrieval_df)

irrt_retrieval_summary_rows = []

for (split_name, model_name, pipeline_name), g in irrt_retrieval_df.groupby(["split", "model", "pipeline"], sort=False):
    row = {
        "split": split_name,
        "model": model_name,
        "pipeline": pipeline_name,
    }

    for m in IRRT_RETRIEVAL_CUTOFFS:
        col = f"CandidateRecall@{m}"
        vals = g[col].astype(float)
        if len(vals) == 1:
            row[col] = f"{float(vals.iloc[0]):.4f}"
        else:
            s = summarize_metric(vals)
            row[col] = f"{s['mean']:.4f} ± {s['ci95']:.4f}"

    irrt_retrieval_summary_rows.append(row)

irrt_retrieval_summary_df = pd.DataFrame(irrt_retrieval_summary_rows).sort_values(
    ["split", "pipeline", "model"], kind="stable"
).reset_index(drop=True)

display(irrt_retrieval_summary_df)

Model-independent sparse retrieval summary:


,split,pipeline,CandidateRecall@50,CandidateRecall@100,CandidateRecall@200
0,valid,SparseLexical,0.123650,0.159626,0.197422
1,test,SparseLexical,0.108293,0.141817,0.177862


,split,model,pipeline,CandidateRecall@50,CandidateRecall@100,CandidateRecall@200
0,test,FMLP-Rec,ExactDenseRetriever,0.0414 ± 0.0033,0.0677 ± 0.0035,0.1086 ± 0.0045
1,test,MetadataComiRecSA,ExactDenseRetriever,0.1614 ± 0.0065,0.2462 ± 0.0071,0.3587 ± 0.0086
2,test,MetadataComiRecSA+LabelAware+DivReg,ExactDenseRetriever,0.0362 ± 0.0169,0.0616 ± 0.0268,0.1021 ± 0.0407
3,test,FMLP-Rec,HybridRRF,0.1038 ± 0.0024,0.1472 ± 0.0031,0.2010 ± 0.0036
4,test,MetadataComiRecSA,HybridRRF,0.1735 ± 0.0049,0.2520 ± 0.0058,0.3462 ± 0.0063
5,test,MetadataComiRecSA+LabelAware+DivReg,HybridRRF,0.0987 ± 0.0094,0.1411 ± 0.0158,0.1935 ± 0.0240
6,test,FMLP-Rec,SparseLexical,0.1083 ± 0.0000,0.1418 ± 0.0000,0.1779 ± 0.0000
7,test,MetadataComiRecSA,SparseLexical,0.1083 ± 0.0000,0.1418 ± 0.0000,0.1779 ± 0.0000
8,test,MetadataComiRecSA+LabelAware+DivReg,SparseLexical,0.1083 ± 0.0000,0.1418 ± 0.0000,0.1779 ± 0.0000
9,valid,FMLP-Rec,ExactDenseRetriever,0.0444 ± 0.0032,0.0718 ± 0.0038,0.1144 ± 0.0047
